In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

In [2]:
import pandas as pd

# Wider pandas display so the wide voting matrix prints fully in the notebook
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)
pd.set_option("display.width", 1000)

# SF1 Audience Results Verification

This notebook reproduces the **Eurovision SF1 audience-voting calculation** end-to-end so the official scoring can be independently verified.

The pipeline is structured in stages:

1. **Stage 1 - Data structure**: load the input CSVs, define static reference data (pots, running order, ISO codes), and run basic data-quality checks.
2. **Stage 2 - Selection & cleanup**: pick which show (SF1 / SF2 / GF) to analyse, aggregate audience votes per country, and zero-out any self-voting.
3. **Stage 3 - Pot calculation**: detect countries below the audience-vote threshold and replace their votes by an equalised pot mean (or by jury votes, as fallback).
4. **Stage 4 - Jury calculation**: convert jury ranks into Eurovision points, apply tie-breaking, compute totals and global rankings.
5. **Stage 5 - Televoting calculation**: convert audience votes to points, apply tie-breaking, and produce the final televoting ranking.
6. **Final results**: combine jury + televoting into the official combined scoreboard and export Excel deliverables.

---

## Stage 1: Create the data structure

What this stage does:

- Import libraries and configure pandas display options.
- Define static reference dictionaries (CSV paths, pots, running order, ISO codes).
- Load all available Jury and Televoting CSVs into dataframes (missing files are skipped).
- Define the audience-vote threshold used in pot calculations.
- Run the first data-quality check: detect duplicated `TPartnerId` values per televoting provider.

In [3]:
import os
import shutil
import stat
import hashlib
import pandas as pd
from sqlalchemy import create_engine
import math
import numpy as np

########################
### STATIC VARIABLES ###
########################

SOURCE_CSV_DIRECTORY = "./Input/ONCE Test Scenario"
SCRIPT_XLSX_DIRECTORY = "./Output"

###########
# SF1 CSV #
###########

SF1_JURY_CSV_FILE_PATH = os.path.join(SOURCE_CSV_DIRECTORY, "Test-01_SF1_EY_Jury.csv")
SF1_TELEVOTING_CSV_FILE_PATH = os.path.join(SOURCE_CSV_DIRECTORY, "Test-01_SF1_EY_Televoting.csv")

###########
# SF2 CSV #
###########

SF2_JURY_CSV_FILE_PATH = os.path.join(SOURCE_CSV_DIRECTORY, "")
SF2_TELEVOTING_CSV_FILE_PATH = os.path.join(SOURCE_CSV_DIRECTORY, "")

##########
# GF CSV #
##########

GF_JURY_CSV_FILE_PATH = os.path.join(SOURCE_CSV_DIRECTORY, "Test-03_GF_EY_Jury.csv")
GF_TELEVOTING_CSV_FILE_PATH = os.path.join(SOURCE_CSV_DIRECTORY, "Test-03_GF_EY_Televoting.csv")

################
# POTS MAPPING #
################

DICT_POTS_SF1 = {
    "Pot01": ["Croatia", "Finland", "Montenegro", "Serbia", "Sweden"],
    "Pot02": ["Belgium", "Georgia", "Israel", "Moldova", "Poland"],
    "Pot03": ["Estonia", "Greece", "Lithuania", "Portugal", "San Marino"],
}
 
DICT_POTS_SF2 = {
    "Pot01": ["Albania", "Australia", "Denmark", "Norway", "Switzerland"],
    "Pot02": ["Armenia", "Azerbaijan", "Luxembourg", "Romania", "Ukraine"],
    "Pot03": ["Bulgaria", "Cyprus", "Czechia", "Latvia", "Malta"],
}
 
DICT_POTS_GF = {
    "Pot01": ["Albania", "Bulgaria", "Croatia", "Montenegro", "Serbia", "Switzerland"],
    "Pot02": ["Australia", "Denmark", "Estonia", "Finland", "Norway", "Sweden"],
    "Pot03": ["Armenia", "Azerbaijan", "Georgia", "Israel", "Poland", "Ukraine"],
    "Pot04": ["Belgium", "Czechia", "Luxembourg", "Moldova", "Portugal", "Romania"],
    "Pot05": ["Cyprus", "Greece", "Latvia", "Lithuania", "Malta", "San Marino"]
}

DICT_POTS_PREQUALIFIED = {
    "Pot00": ["Austria", "France", "Germany", "Italy", "United Kingdom"]
}
 
# PDF page 25: "The RoW is not considered when calculating substitution results."
# Therefore RoW is excluded from the Pot 0 donor pool.
LIST_POT_00_PREQUALIFIED = ["Austria", "France", "Germany", "Italy", "United Kingdom"]

#########################
# RUNNING ORDER MAPPING #
#########################

DICT_DDI_SF1 = {
    "DDI01": "Moldova",
    "DDI02": "Sweden",
    "DDI03": "Croatia",
    "DDI04": "Greece",
    "DDI05": "Portugal",
    "DDI06": "Georgia",
    "DDI07": "Finland",
    "DDI08": "Montenegro",
    "DDI09": "Estonia",
    "DDI10": "Israel",
    "DDI11": "Belgium",
    "DDI12": "Lithuania",
    "DDI13": "San Marino",
    "DDI14": "Poland",
    "DDI15": "Serbia"
}

DICT_DDI_SF2 = {
    "DDI01": "Bulgaria",
    "DDI02": "Azerbaijan",
    "DDI03": "Romania",
    "DDI04": "Luxembourg",
    "DDI05": "Czechia",
    "DDI06": "Armenia",
    "DDI07": "Switzerland",
    "DDI08": "Cyprus",
    "DDI09": "Latvia",
    "DDI10": "Denmark",
    "DDI11": "Australia",
    "DDI12": "Ukraine",
    "DDI13": "Albania",
    "DDI14": "Malta",
    "DDI15": "Norway"
}

# fake mapping for ONCE test scenario
DICT_DDI_GF = {
    "DDI01": "Australia",
    "DDI02": "Austria",
    "DDI03": "Cyprus",
    "DDI04": "Denmark",
    "DDI05": "Latvia",
    "DDI06": "Luxembourg",
    "DDI07": "Norway",
    "DDI08": "Romania",
    "DDI09": "Ukraine",
    "DDI10": "Czechia",
    "DDI11": "Switzerland",
    "DDI12": "France",
    "DDI13": "United Kingdom",
    "DDI14": "Belgium",
    "DDI15": "Estonia",
    "DDI16": "Finland",
    "DDI17": "Greece",
    "DDI18": "Lithuania",
    "DDI19": "Germany",
    "DDI20": "Italy",
    "DDI21": "Poland",
    "DDI22": "Portugal",
    "DDI23": "Sweden",
    "DDI24": "Serbia",
    "DDI25": "Moldova"
}

DICT_ISO = {
    "Albania": "AL",
    "Andorra": "AD",
    "Armenia": "AM",
    "Australia": "AU",
    "Austria": "AT",
    "Azerbaijan": "AZ",
    "Belarus": "BY",
    "Belgium": "BE",
    "Bosnia & Herzegovina": "BA",
    "Bulgaria": "BG",
    "Croatia": "HR",
    "Cyprus": "CY",
    "Czechia": "CZ",
    "Denmark": "DK",
    "Estonia": "EE",
    "Finland": "FI",
    "France": "FR",
    "Georgia": "GE",
    "Germany": "DE",
    "Greece": "GR",
    "Hungary": "HU",
    "Iceland": "IS",
    "Ireland": "IE",
    "Israel": "IL",
    "Italy": "IT",
    "Latvia": "LV",
    "Lithuania": "LT",
    "Luxembourg": "LU",
    "Malta": "MT",
    "Moldova": "MD",
    "Monaco": "MC",
    "Montenegro": "ME",
    "Morocco": "MA",
    "Netherlands": "NL",
    "North Macedonia": "MK",
    "Norway": "NO",
    "Poland": "PL",
    "Portugal": "PT",
    "Romania": "RO",
    "Russia": "RU",
    "San Marino": "SM",
    "Serbia": "RS",
    "Slovakia": "SK",
    "Slovenia": "SI",
    "Spain": "ES",
    "Sweden": "SE",
    "Switzerland": "CH",
    "Turkey": "TR",
    "Ukraine": "UA",
    "United Kingdom": "GB",
    "Rest Of World": "RoW",
}

MAPPING_DF_TO_DICT_DDI = {
    "sf1_jury_df": DICT_DDI_SF1,
    "sf1_televoting_df": DICT_DDI_SF1,
    "sf2_jury_df": DICT_DDI_SF2,
    "sf2_televoting_df": DICT_DDI_SF2,
    "gf_jury_df": DICT_DDI_GF,
    "gf_televoting_df": DICT_DDI_GF,
}

MAPPING_DF_TO_DICT_POTS = {
    "sf1_jury_df": DICT_POTS_SF1,
    "sf1_televoting_df": DICT_POTS_SF1,
    "sf2_jury_df": DICT_POTS_SF2,
    "sf2_televoting_df": DICT_POTS_SF2,
    "gf_jury_df": DICT_POTS_GF,
    "gf_televoting_df": DICT_POTS_GF,
}

POT_VOTE_COUNT_THRESHOLD = 1000
# Pot-substituted audience values are rounded to the nearest multiple of
# POT_ROUND_TO before ranking, so close averages get tie-broken by jury rank.
POT_SCALE_UP = 15  # scale norm_mean before rounding, then keeps integer arithmetic
POT_ROUND_TO = 3  # rounding bucket for pot-substituted audience

# list of countries to replace audience voting by jury voting
list_replace_audience_by_jury = []

#########################
### GENERAL FUNCTIONS ###
#########################

# Lists all user-created dataframes
def get_all_dataframe_names():
    dataframe_names = []
    for name, obj in globals().items():
        if isinstance(obj, pd.DataFrame):
            dataframe_names.append(name)
    return dataframe_names

####################
### QC FUNCTIONS ###
####################

def qc01_duplicated_tpartnerid_values():
    # Get the list of all user-created DataFrames
    user_created_dataframe_names = get_all_dataframe_names()

    # list of dataframes that are expected to contain the TPartnerId field (televoting only)
    televoting_dataframes = [
        "sf1_televoting_df",
        "sf2_televoting_df",
        "gf_televoting_df",
    ]

    for df_name in user_created_dataframe_names:
        if df_name in televoting_dataframes:
            df = eval(
                df_name
            )  # load dataframe into temporary variable based on original variable name
            duplicated_rows = df[
                df.duplicated(subset=["strName", "TPartnerId"], keep=False)
            ]

            if len(duplicated_rows) > 0:
                print(
                    f"\n### ERROR, {len(duplicated_rows)} duplicated [TPartnerId] values identified in dataframe {df_name}"
                )
                print(duplicated_rows[["strName", "TPartnerId"]])
            elif len(duplicated_rows) == 0:
                print(
                    f"\nNo duplicated [TPartnerId] values identified in dataframe {df_name}"
                )
            else:
                raise ValueError(
                    f"Unexpected value during qc01_duplicated_tpartnerid_values for dataframe {df_name}."
                )

#########################
### MAIN SCRIPT START ###
#########################

# Store all provided CSVs to dataframes
try:
    sf1_jury_df = pd.read_csv(SF1_JURY_CSV_FILE_PATH, sep=";")
except (FileNotFoundError, IsADirectoryError):
    print("\n### Missing SF1 Jury CSV file, skipping...")
try:
    sf1_televoting_df = pd.read_csv(SF1_TELEVOTING_CSV_FILE_PATH, sep=";")
except:
    print("\n### Missing SF1 Televoting CSV file, skipping...")
try:
    sf2_jury_df = pd.read_csv(SF2_JURY_CSV_FILE_PATH, sep=";")
except (FileNotFoundError, IsADirectoryError):
    print("\n### Missing SF2 Jury CSV file, skipping...")
try:
    sf2_televoting_df = pd.read_csv(SF2_TELEVOTING_CSV_FILE_PATH, sep=";")
except (FileNotFoundError, IsADirectoryError):
    print("\n### Missing SF2 Televoting CSV file, skipping...")
try:
    gf_jury_df = pd.read_csv(GF_JURY_CSV_FILE_PATH, sep=";")
except (FileNotFoundError, IsADirectoryError):
    print("\n### Missing GF Jury CSV file, skipping...")
try:
    gf_televoting_df = pd.read_csv(GF_TELEVOTING_CSV_FILE_PATH, sep=";")
except (FileNotFoundError, IsADirectoryError):
    print("\n### Missing GF Televoting CSV file, skipping...")

# Normalize 'Rest of World' (lowercase 'o') to 'Rest Of World' (uppercase 'O')
# in all jury dataframes. The jury CSVs use the lowercase form while the
# televoting flow uses the uppercase form via DICT_ISO. Without normalization
# this causes two separate RoW columns in the final merged results.
def _normalize_row_casing(df):
    if df is None: return df
    if "strName" in df.columns:
        df["strName"] = df["strName"].replace({"Rest of World": "Rest Of World"})
    return df

for _name in ("sf1_jury_df", "sf2_jury_df", "gf_jury_df"):
    if _name in dir():
        _df = eval(_name)
        _normalize_row_casing(_df)
    
qc01_duplicated_tpartnerid_values()


### Missing SF1 Jury CSV file, skipping...

### Missing SF1 Televoting CSV file, skipping...

### Missing SF2 Jury CSV file, skipping...

### Missing SF2 Televoting CSV file, skipping...

No duplicated [TPartnerId] values identified in dataframe gf_televoting_df


---

---

## Stage 2a: Selecting the dataset to analyse

Pick which show the Rest Of the notebook should run for (SF1 / SF2 / Grand Final televoting dataframe). The selected name drives:

- which DDI-to-country mapping is used,
- which pot mapping is used,
- which Jury dataframe is paired with the televoting one downstream.

In [4]:
###############################
### SELECTING DF TO ANALYZE ###
###############################

########################################
# selected_df_name = "sf1_televoting_df"
# selected_df_name = 'sf2_televoting_df'
selected_df_name = 'gf_televoting_df'
########################################

selected_df = eval(selected_df_name)

# Identify corresponding dictionary
corresponding_ddi_dict = MAPPING_DF_TO_DICT_DDI[selected_df_name]
corresponding_pots_dict = MAPPING_DF_TO_DICT_POTS[selected_df_name]

print(
    f"\nAnalysis running for dataframe =>***{selected_df_name.upper()}***<=\n\nDDI mapping: {corresponding_ddi_dict}\n\n"
)


Analysis running for dataframe =>***GF_TELEVOTING_DF***<=

DDI mapping: {'DDI01': 'Australia', 'DDI02': 'Austria', 'DDI03': 'Cyprus', 'DDI04': 'Denmark', 'DDI05': 'Latvia', 'DDI06': 'Luxembourg', 'DDI07': 'Norway', 'DDI08': 'Romania', 'DDI09': 'Ukraine', 'DDI10': 'Czechia', 'DDI11': 'Switzerland', 'DDI12': 'France', 'DDI13': 'United Kingdom', 'DDI14': 'Belgium', 'DDI15': 'Estonia', 'DDI16': 'Finland', 'DDI17': 'Greece', 'DDI18': 'Lithuania', 'DDI19': 'Germany', 'DDI20': 'Italy', 'DDI21': 'Poland', 'DDI22': 'Portugal', 'DDI23': 'Sweden', 'DDI24': 'Serbia', 'DDI25': 'Moldova'}




---

---

### Aggregating audience votes per country

Group the raw televoting rows by `strName` (country code) and sum every `nVotesforDDI*` column, so that we end up with one aggregated row per voting country.

In [5]:
# Keep only the per-DDI vote columns
filtered_columns = [col for col in selected_df.columns if col.startswith("nVotesforDDI")]

# Group rows by country code (strName) and sum the per-DDI vote counts
grouped_df = selected_df.groupby("strName")[filtered_columns].sum()

# Re-add the partner id placeholder for downstream consistency
grouped_df.insert(0, "TPartnerId", "AGGREGATED")
grouped_df = grouped_df.reset_index(drop=False)

grouped_df

,strName,TPartnerId,nVotesforDDI01,nVotesforDDI02,nVotesforDDI03,nVotesforDDI04,nVotesforDDI05,nVotesforDDI06,nVotesforDDI07,nVotesforDDI08,nVotesforDDI09,nVotesforDDI10,nVotesforDDI11,nVotesforDDI12,nVotesforDDI13,nVotesforDDI14,nVotesforDDI15,nVotesforDDI16,nVotesforDDI17,nVotesforDDI18,nVotesforDDI19,nVotesforDDI20,nVotesforDDI21,nVotesforDDI22,nVotesforDDI23,nVotesforDDI24,nVotesforDDI25
0,AL,AGGREGATED,2745,2975,2751,2653,2864,1562,1642,2286,2628,2014,3315,2697,3074,2761,2184,3205,2063,1407,1748,2028,1913,1928,3230,1478,1934
1,AM,AGGREGATED,13637,14167,12338,14883,12723,12335,13332,12982,13713,12990,12327,14979,12817,14870,14042,14333,13496,12513,12330,14087,14705,14828,15056,12228,12829
2,AT,AGGREGATED,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,AU,AGGREGATED,0,152810,214910,142699,240141,131170,108562,108229,194951,125982,179682,151464,240796,90344,76179,179905,198502,80929,210281,106905,80970,184074,172574,230330,138253
4,AZ,AGGREGATED,14965,17025,8363,17022,11960,7783,14329,16326,12869,14033,9998,10803,16781,13951,10040,9323,10839,16493,8670,16650,6950,18827,10398,14922,12457
5,BE,AGGREGATED,186742,221262,199496,210520,205557,152876,215708,169468,234311,171969,216375,192440,244067,0,170977,178270,187943,167301,223256,170402,158232,184688,201765,166017,180535
6,BG,AGGREGATED,22229,19455,24175,19323,22720,24252,20163,17395,20718,24144,21895,22950,19303,19246,22443,18690,18068,22314,20122,24068,18582,21990,24825,23107,21879
7,CH,AGGREGATED,133644,100486,204649,119876,138497,219271,201212,197879,214523,170845,0,208215,113158,123800,228186,205737,200842,165867,219181,111724,140816,231986,155436,196203,213295
8,CY,AGGREGATED,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,CZ,AGGREGATED,117357,56268,34651,138743,141058,111105,102436,101656,80105,0,130970,109826,106443,136655,92590,102766,48924,113966,114385,58796,80214,137224,75197,95319,61229


---

---

### Adding readable country names

- Map the ISO code in `strName` to the full country name (`FullCountryName`).
- Append the recipient country name to each `nVotesforDDI##` column header so the matrix becomes self-describing.

In [6]:
# 1) Add the full country name resolved from the ISO code (strName)
inverted_DICT_ISO = {v: k for k, v in DICT_ISO.items()}
grouped_df_w_full_country_names = grouped_df.assign(
    FullCountryName=grouped_df["strName"].map(inverted_DICT_ISO)
)
grouped_df_w_full_country_names.insert(
    0, "FullCountryName", grouped_df_w_full_country_names.pop("FullCountryName")
)

# 2) Append the recipient country name to each nVotesforDDI## column header,
#    so a column reads e.g. nVotesforDDI01_Moldova instead of nVotesforDDI01.
for ddi_key, country_name in corresponding_ddi_dict.items():
    ddi_number = ddi_key[-2:]
    column_to_update = [
        col for col in grouped_df_w_full_country_names.columns
        if col.startswith(f"nVotesforDDI{ddi_number}")
    ][0]
    new_column_name = f"{column_to_update}_{country_name}"
    grouped_df_w_full_country_names = grouped_df_w_full_country_names.rename(
        columns={column_to_update: new_column_name}
    )

grouped_df_w_full_country_names

,FullCountryName,strName,TPartnerId,nVotesforDDI01_Australia,nVotesforDDI02_Austria,nVotesforDDI03_Cyprus,nVotesforDDI04_Denmark,nVotesforDDI05_Latvia,nVotesforDDI06_Luxembourg,nVotesforDDI07_Norway,nVotesforDDI08_Romania,nVotesforDDI09_Ukraine,nVotesforDDI10_Czechia,nVotesforDDI11_Switzerland,nVotesforDDI12_France,nVotesforDDI13_United Kingdom,nVotesforDDI14_Belgium,nVotesforDDI15_Estonia,nVotesforDDI16_Finland,nVotesforDDI17_Greece,nVotesforDDI18_Lithuania,nVotesforDDI19_Germany,nVotesforDDI20_Italy,nVotesforDDI21_Poland,nVotesforDDI22_Portugal,nVotesforDDI23_Sweden,nVotesforDDI24_Serbia,nVotesforDDI25_Moldova
0,Albania,AL,AGGREGATED,2745,2975,2751,2653,2864,1562,1642,2286,2628,2014,3315,2697,3074,2761,2184,3205,2063,1407,1748,2028,1913,1928,3230,1478,1934
1,Armenia,AM,AGGREGATED,13637,14167,12338,14883,12723,12335,13332,12982,13713,12990,12327,14979,12817,14870,14042,14333,13496,12513,12330,14087,14705,14828,15056,12228,12829
2,Austria,AT,AGGREGATED,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,Australia,AU,AGGREGATED,0,152810,214910,142699,240141,131170,108562,108229,194951,125982,179682,151464,240796,90344,76179,179905,198502,80929,210281,106905,80970,184074,172574,230330,138253
4,Azerbaijan,AZ,AGGREGATED,14965,17025,8363,17022,11960,7783,14329,16326,12869,14033,9998,10803,16781,13951,10040,9323,10839,16493,8670,16650,6950,18827,10398,14922,12457
5,Belgium,BE,AGGREGATED,186742,221262,199496,210520,205557,152876,215708,169468,234311,171969,216375,192440,244067,0,170977,178270,187943,167301,223256,170402,158232,184688,201765,166017,180535
6,Bulgaria,BG,AGGREGATED,22229,19455,24175,19323,22720,24252,20163,17395,20718,24144,21895,22950,19303,19246,22443,18690,18068,22314,20122,24068,18582,21990,24825,23107,21879
7,Switzerland,CH,AGGREGATED,133644,100486,204649,119876,138497,219271,201212,197879,214523,170845,0,208215,113158,123800,228186,205737,200842,165867,219181,111724,140816,231986,155436,196203,213295
8,Cyprus,CY,AGGREGATED,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,Czechia,CZ,AGGREGATED,117357,56268,34651,138743,141058,111105,102436,101656,80105,0,130970,109826,106443,136655,92590,102766,48924,113966,114385,58796,80214,137224,75197,95319,61229


---

---

### Sanity check: full country names populated

If any `FullCountryName` remained `NaN` after the ISO mapping (e.g. for `Rest Of World`), fall back to the original `strName` value so no row is left without a label.

In [7]:
# Fallback: if any FullCountryName is missing, use the original ISO code as label
if any(grouped_df_w_full_country_names["FullCountryName"].isna()):
    grouped_df_w_full_country_names["FullCountryName"] = grouped_df_w_full_country_names["strName"]

---

---

### Detecting and neutralising self-voting

A country is not allowed to vote for itself. We scan the matrix and force any non-zero self-voting cell to `0`, printing an error so the issue is visible in the run log.

In [8]:
# A country must not vote for itself: any non-zero self-vote cell is reset to 0
for index, row in grouped_df_w_full_country_names.iterrows():
    full_country_name = row["FullCountryName"]
    matching_column = [
        col for col in grouped_df_w_full_country_names.columns
        if col.startswith("nVotesforDDI") and col.endswith(full_country_name)
    ]

    if matching_column:
        current_value = grouped_df_w_full_country_names.at[index, matching_column[0]]
        if current_value != 0:
            print(f"### ERROR: SELF-VOTING COUNTRY IDENTIFIED! Country: {full_country_name} - Value: {current_value}")
            grouped_df_w_full_country_names.at[index, matching_column[0]] = 0

grouped_df_w_full_country_names

,FullCountryName,strName,TPartnerId,nVotesforDDI01_Australia,nVotesforDDI02_Austria,nVotesforDDI03_Cyprus,nVotesforDDI04_Denmark,nVotesforDDI05_Latvia,nVotesforDDI06_Luxembourg,nVotesforDDI07_Norway,nVotesforDDI08_Romania,nVotesforDDI09_Ukraine,nVotesforDDI10_Czechia,nVotesforDDI11_Switzerland,nVotesforDDI12_France,nVotesforDDI13_United Kingdom,nVotesforDDI14_Belgium,nVotesforDDI15_Estonia,nVotesforDDI16_Finland,nVotesforDDI17_Greece,nVotesforDDI18_Lithuania,nVotesforDDI19_Germany,nVotesforDDI20_Italy,nVotesforDDI21_Poland,nVotesforDDI22_Portugal,nVotesforDDI23_Sweden,nVotesforDDI24_Serbia,nVotesforDDI25_Moldova
0,Albania,AL,AGGREGATED,2745,2975,2751,2653,2864,1562,1642,2286,2628,2014,3315,2697,3074,2761,2184,3205,2063,1407,1748,2028,1913,1928,3230,1478,1934
1,Armenia,AM,AGGREGATED,13637,14167,12338,14883,12723,12335,13332,12982,13713,12990,12327,14979,12817,14870,14042,14333,13496,12513,12330,14087,14705,14828,15056,12228,12829
2,Austria,AT,AGGREGATED,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,Australia,AU,AGGREGATED,0,152810,214910,142699,240141,131170,108562,108229,194951,125982,179682,151464,240796,90344,76179,179905,198502,80929,210281,106905,80970,184074,172574,230330,138253
4,Azerbaijan,AZ,AGGREGATED,14965,17025,8363,17022,11960,7783,14329,16326,12869,14033,9998,10803,16781,13951,10040,9323,10839,16493,8670,16650,6950,18827,10398,14922,12457
5,Belgium,BE,AGGREGATED,186742,221262,199496,210520,205557,152876,215708,169468,234311,171969,216375,192440,244067,0,170977,178270,187943,167301,223256,170402,158232,184688,201765,166017,180535
6,Bulgaria,BG,AGGREGATED,22229,19455,24175,19323,22720,24252,20163,17395,20718,24144,21895,22950,19303,19246,22443,18690,18068,22314,20122,24068,18582,21990,24825,23107,21879
7,Switzerland,CH,AGGREGATED,133644,100486,204649,119876,138497,219271,201212,197879,214523,170845,0,208215,113158,123800,228186,205737,200842,165867,219181,111724,140816,231986,155436,196203,213295
8,Cyprus,CY,AGGREGATED,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,Czechia,CZ,AGGREGATED,117357,56268,34651,138743,141058,111105,102436,101656,80105,0,130970,109826,106443,136655,92590,102766,48924,113966,114385,58796,80214,137224,75197,95319,61229


---

---

### Assigning a Pot to each voting country

Every country is mapped to its allocated pot (`Pot01`-`Pot03` for SF1, plus `Pot00` for the prequalified Big Five / host).

### Reference: list of pre-qualified countries (Pot00)

Quick visual check of the prequalified country list.


In [9]:
LIST_POT_00_PREQUALIFIED

['Austria', 'France', 'Germany', 'Italy', 'United Kingdom']

In [10]:
# Build a country -> pot mapping from the active pot dictionary
inverted_DICT_POTS = {
    country: pot
    for pot, countries in corresponding_pots_dict.items()
    for country in countries
}
pot_df = grouped_df_w_full_country_names

# Add (and reposition) the POT column
pot_df["POT"] = pot_df["FullCountryName"].map(inverted_DICT_POTS)
pot_df.insert(2, "POT", pot_df.pop("POT"))

# Pre-qualified Big Five + host country are placed in Pot00
pot_df.loc[pot_df["FullCountryName"].isin(LIST_POT_00_PREQUALIFIED), "POT"] = "Pot00"

pot_df

,FullCountryName,strName,POT,TPartnerId,nVotesforDDI01_Australia,nVotesforDDI02_Austria,nVotesforDDI03_Cyprus,nVotesforDDI04_Denmark,nVotesforDDI05_Latvia,nVotesforDDI06_Luxembourg,nVotesforDDI07_Norway,nVotesforDDI08_Romania,nVotesforDDI09_Ukraine,nVotesforDDI10_Czechia,nVotesforDDI11_Switzerland,nVotesforDDI12_France,nVotesforDDI13_United Kingdom,nVotesforDDI14_Belgium,nVotesforDDI15_Estonia,nVotesforDDI16_Finland,nVotesforDDI17_Greece,nVotesforDDI18_Lithuania,nVotesforDDI19_Germany,nVotesforDDI20_Italy,nVotesforDDI21_Poland,nVotesforDDI22_Portugal,nVotesforDDI23_Sweden,nVotesforDDI24_Serbia,nVotesforDDI25_Moldova
0,Albania,AL,Pot01,AGGREGATED,2745,2975,2751,2653,2864,1562,1642,2286,2628,2014,3315,2697,3074,2761,2184,3205,2063,1407,1748,2028,1913,1928,3230,1478,1934
1,Armenia,AM,Pot03,AGGREGATED,13637,14167,12338,14883,12723,12335,13332,12982,13713,12990,12327,14979,12817,14870,14042,14333,13496,12513,12330,14087,14705,14828,15056,12228,12829
2,Austria,AT,Pot00,AGGREGATED,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,Australia,AU,Pot02,AGGREGATED,0,152810,214910,142699,240141,131170,108562,108229,194951,125982,179682,151464,240796,90344,76179,179905,198502,80929,210281,106905,80970,184074,172574,230330,138253
4,Azerbaijan,AZ,Pot03,AGGREGATED,14965,17025,8363,17022,11960,7783,14329,16326,12869,14033,9998,10803,16781,13951,10040,9323,10839,16493,8670,16650,6950,18827,10398,14922,12457
5,Belgium,BE,Pot04,AGGREGATED,186742,221262,199496,210520,205557,152876,215708,169468,234311,171969,216375,192440,244067,0,170977,178270,187943,167301,223256,170402,158232,184688,201765,166017,180535
6,Bulgaria,BG,Pot01,AGGREGATED,22229,19455,24175,19323,22720,24252,20163,17395,20718,24144,21895,22950,19303,19246,22443,18690,18068,22314,20122,24068,18582,21990,24825,23107,21879
7,Switzerland,CH,Pot01,AGGREGATED,133644,100486,204649,119876,138497,219271,201212,197879,214523,170845,0,208215,113158,123800,228186,205737,200842,165867,219181,111724,140816,231986,155436,196203,213295
8,Cyprus,CY,Pot05,AGGREGATED,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,Czechia,CZ,Pot04,AGGREGATED,117357,56268,34651,138743,141058,111105,102436,101656,80105,0,130970,109826,106443,136655,92590,102766,48924,113966,114385,58796,80214,137224,75197,95319,61229


---

## Stage 3: Pot threshold check

For each voting country, compute the total number of audience votes sent (`TOTAL_SENT_VOTES`) and check whether it meets the minimum threshold (`POT_VOTE_COUNT_THRESHOLD`).

- If **all** countries are above the threshold, no pot replacement is required and the workbook can jump straight to the jury voting stage.
- If **at least one** country falls below the threshold, the next stage will recompute its votes from the pot.

### Total sent votes & threshold flag

For each country, sum every `nVotesforDDI*` column to get the total number of audience votes sent and compare it to the threshold.


In [11]:
# Sum total audience votes sent by each country
votes_columns = [col for col in pot_df.columns if col.startswith("nVotesforDDI")]
pot_df["TOTAL_SENT_VOTES"] = pot_df[votes_columns].sum(axis=1)

# Flag countries that meet the minimum threshold
pot_df["ABOVE_POT_THRESHOLD"] = pot_df["TOTAL_SENT_VOTES"] >= POT_VOTE_COUNT_THRESHOLD

# Decide whether the pot-replacement stage is required at all
if False in pot_df["ABOVE_POT_THRESHOLD"].values:
    required_pot_calculation = True
    print("At least one country does not meet the vote threshold value, triggering pot calculations...")
    threshold_df = pot_df
else:
    required_pot_calculation = False
    print("All countries above threshold vote limit, no pot calculations required.")
    print('Go to cell containing "### FIRST STEP OF JURY VOTING ###"')

# Sort and persist the pre-pot snapshot
pot_df = pot_df.sort_values(["POT", "TOTAL_SENT_VOTES", "FullCountryName"])
pot_df.to_excel(
    os.path.join(SCRIPT_XLSX_DIRECTORY, "01.prePot_Calculations_voting_matrix.xlsx"),
    index=False,
)

pot_df

At least one country does not meet the vote threshold value, triggering pot calculations...


,FullCountryName,strName,POT,TPartnerId,nVotesforDDI01_Australia,nVotesforDDI02_Austria,nVotesforDDI03_Cyprus,nVotesforDDI04_Denmark,nVotesforDDI05_Latvia,nVotesforDDI06_Luxembourg,nVotesforDDI07_Norway,nVotesforDDI08_Romania,nVotesforDDI09_Ukraine,nVotesforDDI10_Czechia,nVotesforDDI11_Switzerland,nVotesforDDI12_France,nVotesforDDI13_United Kingdom,nVotesforDDI14_Belgium,nVotesforDDI15_Estonia,nVotesforDDI16_Finland,nVotesforDDI17_Greece,nVotesforDDI18_Lithuania,nVotesforDDI19_Germany,nVotesforDDI20_Italy,nVotesforDDI21_Poland,nVotesforDDI22_Portugal,nVotesforDDI23_Sweden,nVotesforDDI24_Serbia,nVotesforDDI25_Moldova,TOTAL_SENT_VOTES,ABOVE_POT_THRESHOLD
2,Austria,AT,Pot00,AGGREGATED,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,False
10,Germany,DE,Pot00,AGGREGATED,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,False
15,United Kingdom,GB,Pot00,AGGREGATED,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,False
20,Italy,IT,Pot00,AGGREGATED,202904,168401,143897,155502,170648,211440,194359,220953,213918,156592,200775,194321,156151,187433,218756,215850,152135,156850,201997,0,144253,195237,157543,130843,133062,4283820,True
14,France,FR,Pot00,AGGREGATED,294776,714353,264052,509591,422889,463412,346891,379371,402521,373523,175674,0,714786,639742,525482,184432,166901,248685,750588,340105,685252,535481,182920,132997,161645,9616069,True
0,Albania,AL,Pot01,AGGREGATED,2745,2975,2751,2653,2864,1562,1642,2286,2628,2014,3315,2697,3074,2761,2184,3205,2063,1407,1748,2028,1913,1928,3230,1478,1934,59085,True
6,Bulgaria,BG,Pot01,AGGREGATED,22229,19455,24175,19323,22720,24252,20163,17395,20718,24144,21895,22950,19303,19246,22443,18690,18068,22314,20122,24068,18582,21990,24825,23107,21879,534056,True
25,Montenegro,ME,Pot01,AGGREGATED,22524,22212,23962,29618,25908,21388,24541,22715,29399,21938,29783,28602,20925,24595,25072,22203,26428,29374,26207,28615,27955,20994,26869,20455,27202,629484,True
18,Croatia,HR,Pot01,AGGREGATED,84936,60041,73128,71802,81572,82681,69784,80594,82993,75228,68185,76610,79555,62539,78812,71472,68956,61785,69751,62473,84999,60852,62894,81851,59791,1813284,True
31,Serbia,RS,Pot01,AGGREGATED,55440,94232,60203,87956,87672,80965,77453,87240,88337,60873,98133,72562,86074,53033,52295,97639,92251,74868,54637,87670,74896,72708,82940,0,56239,1836316,True


---

## Stage 3a: Pot replacement calculation

For each country below the threshold:

1. `get_pots(...)` decides whether to use **audience** votes from the same pot (and Pot00) or fall back to **jury** votes.
2. `calcul_replacement(...)` and `calcul_replacement_no_votes(...)` compute the equalised pot mean that will replace the country's audience vote breakdown.
3. The result is written back into `threshold_df` and exported to per-country Excel files.

### Helper: `get_pots`

Decide whether a country falls back to **audience** or **jury** votes, and return the dataframe of donor countries used to recompute the pot mean.


In [12]:
def get_pots(country_name, df_all_countries):
    """For a country below the threshold, decide which votes to use as replacement.

    Returns a tuple ``(mode, df_used)`` where ``mode`` is either ``'audience'``
    or ``'jury'`` and ``df_used`` is the dataframe of donor countries used to
    recompute the equalised pot mean (empty string when ``mode == 'jury'``).
    """
    pot = df_all_countries.loc[
        df_all_countries["FullCountryName"] == country_name, "POT"
    ].unique()[0]

    same_pot = df_all_countries[df_all_countries["POT"] == pot]
    pot_0 = df_all_countries[df_all_countries["POT"] == "Pot00"]

    same_pot_above_t = same_pot.loc[same_pot["ABOVE_POT_THRESHOLD"] == True]
    pot_0_above_t = pot_0.loc[pot_0["ABOVE_POT_THRESHOLD"] == True]

    # Pot00 (prequalified countries) follows its own rules
    if pot == "Pot00":
        if len(pot_0_above_t) > 1:
            return "audience", pot_0_above_t
        return "jury", ""

    # Standard SF pots
    if len(same_pot_above_t) >= 2:
        return "audience", same_pot_above_t
    if (len(same_pot_above_t) == 1) and (len(pot_0_above_t) > 0):
        return "audience", pd.concat([same_pot_above_t, pot_0_above_t])
    if (len(same_pot_above_t) == 1) and (len(pot_0_above_t) == 0):
        return "jury", ""
    if (len(same_pot_above_t) == 0) and (len(pot_0_above_t) > 1):
        return "audience", pd.concat([same_pot_above_t, pot_0_above_t])
    if (len(same_pot_above_t) == 0) and (len(pot_0_above_t) <= 1):
        return "jury", ""

    print("Something weird happened")
    return "", ""

### Helper: `calcul_replacement`

Equalised-pot calculation when the country **did** send some votes.


In [13]:
def calcul_replacement(country, tmp_df_pot,points_columns,country_name ):
    print('Calculating pot votes replacement for ', country_name)
    
    # Calculate total votes per donor and equalise (Step 2 of manual Excel: 
    # each donor's vote / their own total → gives a proportion 0..1).
    tmp_df_pot.insert(0,"total_votes",tmp_df_pot[points_columns].sum(axis=1))
    tmp_df_pot.insert(0,"factor",tmp_df_pot["total_votes"])
    
    # Calculate equalised votes (proportions per donor)
    tmp_df_pot2 = tmp_df_pot[points_columns].div(tmp_df_pot["factor"], axis=0)
    tmp_df_pot2 = pd.DataFrame(tmp_df_pot2, columns=points_columns, index=tmp_df_pot.index)
    tmp_df_pot2['FullCountryName'] = tmp_df_pot['FullCountryName']
    
    # Prepare new dataframe 
    mean_values = tmp_df_pot2.drop('FullCountryName', axis=1).mean()
    new_row_data = mean_values.to_dict()
    new_row_data['FullCountryName'] = 'mean'
    tmp_df_pot3 = pd.concat([tmp_df_pot2,pd.DataFrame([new_row_data])], ignore_index=True)
    
    # Calculate mask and means 
    means = {}
    for col in points_columns:
        country_suffix = col.replace('nVotesforDDI', '').split('_')[-1]
        mask = ~tmp_df_pot2['FullCountryName'].str.endswith(country_suffix)
        means[col] = tmp_df_pot2.loc[mask, col].mean()
        mean_index = tmp_df_pot3[tmp_df_pot3['FullCountryName'] == 'mean'].index
        if country_suffix.replace(" ", "") == country_name.replace(" ", ""):
            tmp_df_pot3.loc[mean_index,col] = 0
        else :
            tmp_df_pot3.loc[mean_index,col] = means[col]

    # Calculate factor 2, and apply it to all votes
    factor2 = (POT_VOTE_COUNT_THRESHOLD-int(tmp_row_country[points_columns].sum(axis=1)))/ tmp_df_pot3.loc[mean_index,points_columns].sum(axis=1)
    new_mean = tmp_df_pot3.loc[mean_index,points_columns] * factor2.values[0]
    new_mean.insert(len(new_mean.columns), "FullCountryName","norm_mean")
    tmp_df_pot3 = pd.concat([tmp_df_pot3,new_mean])
    
    # Save pot calculation to Excel
    name_excel = 'pot_calculation_for_' + str(country_name) + '.xlsx'
    tmp_df_pot3.to_excel(os.path.join(
    SCRIPT_XLSX_DIRECTORY, name_excel
    ), index=False)
    
    return tmp_df_pot3

### Helper: `calcul_replacement_no_votes`

Equalised-pot calculation when the country sent **no** votes at all.


In [14]:
def calcul_replacement_no_votes(country, tmp_df_pot,points_columns,country_name):
    print('Calculating pot votes replacement for ', country_name, ' with no votes delivered process.')
    
     # Calculate total votes and first factor
    tmp_df_pot.insert(0,"total_votes",tmp_df_pot[points_columns].sum(axis=1))
    tmp_df_pot.insert(0,"factor",tmp_df_pot[points_columns].sum(axis=1))
    
    # Calculate equlized votes
    tmp_df_pot2 = tmp_df_pot[points_columns].div(tmp_df_pot["factor"], axis=0)
    tmp_df_pot2 = pd.DataFrame(tmp_df_pot2, columns=points_columns, index=tmp_df_pot.index)
    tmp_df_pot2['FullCountryName'] = tmp_df_pot['FullCountryName']
    
     # Prepare new dataframe 
    mean_values = tmp_df_pot2.drop('FullCountryName', axis=1).mean()
    new_row_data = mean_values.to_dict()
    new_row_data['FullCountryName'] = 'mean'
    tmp_df_pot3 = pd.concat([tmp_df_pot2,pd.DataFrame([new_row_data])], ignore_index=True)
    
     # Calculate mask and means 
    means = {}
    for col in points_columns:
        country_suffix = col.replace('nVotesforDDI', '').split('_')[-1]
        mask = ~tmp_df_pot2['FullCountryName'].str.endswith(country_suffix)
        means[col] = tmp_df_pot2.loc[mask, col].mean()
        mean_index = tmp_df_pot3[tmp_df_pot3['FullCountryName'] == 'mean'].index
        if country_suffix.replace(" ", "") == country_name.replace(" ", ""):
            tmp_df_pot3.loc[mean_index,col] = 0
        else :
            tmp_df_pot3.loc[mean_index,col] = means[col]
            
    # Calculate factor 2, and apply it to all votes
    # Per the manual Excel calculation: factor2 = (THRESHOLD - voter_actual_votes) / SUM_of_means
    # When the voter has zero actual votes (the SM-style case), this simplifies to
    # THRESHOLD / SUM_of_means rather than just THRESHOLD. The division by SUM_of_means
    # is what makes the resulting "votes" sum to THRESHOLD overall.
    voter_actual = int(tmp_row_country[points_columns].sum(axis=1).iloc[0]) if hasattr(tmp_row_country[points_columns].sum(axis=1), 'iloc') else int(tmp_row_country[points_columns].sum(axis=1))
    sum_of_means = tmp_df_pot3.loc[mean_index, points_columns].sum(axis=1).values[0]
    factor2 = (POT_VOTE_COUNT_THRESHOLD - voter_actual) / sum_of_means
    new_mean = tmp_df_pot3.loc[mean_index,points_columns] * factor2
    new_mean.insert(len(new_mean.columns), "FullCountryName","norm_mean")
    tmp_df_pot3 = pd.concat([tmp_df_pot3,new_mean])
    
    # Save pot calculation to Excel
    name_excel = 'pot_calculation_for_' + str(country_name) + '.xlsx'
    tmp_df_pot3.to_excel(os.path.join(
    SCRIPT_XLSX_DIRECTORY, name_excel
    ), index=False)
    
    return tmp_df_pot3

### Apply pot replacement to all countries below threshold


In [15]:
threshold_df = pot_df

if required_pot_calculation:
    
    list_under_countries = threshold_df.loc[threshold_df["ABOVE_POT_THRESHOLD"] == False,"FullCountryName"].unique()
    points_columns = [col for col in threshold_df.columns if col.startswith('nVotesfor')]
    # Cast vote columns to float so we can store fractional pot-substitute values
    # (PDF §3.3 calls for an "average and equalized" value, not necessarily integer).
    threshold_df[points_columns] = threshold_df[points_columns].astype(float)
    
    # Deal with all countries below threshold
    for country in list_under_countries:
        
        # Take all case into account, say if we want pot calculation or jury voting. Give voting to use
        mode_, df_used = get_pots(country, threshold_df)
        print(country," : " ,mode_)
        
        
        # San Marino (PDF §1.6): no audience vote is conducted, so always use the pot result
        # in semi-finals regardless of what get_pots() returned.
        if (country.replace(" ", "") == 'SanMarino') and (selected_df_name != 'gf_televoting_df'):
            print(f"San Marino — forcing audience-pot calculation per PDF §1.6.")
            if mode_ != 'audience':
                # If get_pots() decided 'jury' (e.g. not enough valid pot members), still
                # build a pot-substitute using whatever donors are available.
                same_pot = threshold_df[threshold_df["POT"] == threshold_df.loc[
                    threshold_df["FullCountryName"] == country, "POT"
                ].unique()[0]]
                pot_0 = threshold_df[threshold_df["POT"] == "Pot00"]
                df_used = pd.concat([
                    same_pot[same_pot["ABOVE_POT_THRESHOLD"] == True],
                    pot_0[pot_0["ABOVE_POT_THRESHOLD"] == True],
                ])
                mode_ = 'audience'

        # Pot calculation with audiences voting of same pot
        if mode_ == 'audience':
            
            # SM-specific: in SF1, SM is a participant (DDI13). To force SM into the
            # calcul_replacement path, we set its votes to 1 except for its own DDI.
            # In GF, SM is NOT a participant, so this block must not fire; SM should
            # take the standard calcul_replacement_no_votes path.
            if country.replace(" ", "") == 'SanMarino' and selected_df_name != 'gf_televoting_df':
                print("Checking San Marino - Audience")
                for col in points_columns:
                    if col != "nVotesforDDI13_San Marino":
                        threshold_df.loc[threshold_df["FullCountryName"] == country,col] = 1
            
            tmp_row_country = threshold_df.loc[threshold_df["FullCountryName"] == country]
            
            if threshold_df.loc[threshold_df["FullCountryName"] == country, "TOTAL_SENT_VOTES"].unique()[0] > 0:
                df = calcul_replacement(tmp_row_country, df_used,points_columns,country)
            else :
                df = calcul_replacement_no_votes(tmp_row_country, df_used,points_columns,country)
            
            #print(country)
            #display(df)
            
            for col in points_columns:
                # Pot-substitute audience per PDF §3.3 and the manual Excel:
                # take the equalised norm_mean and apply CEIL (matching the Excel's "Rounded" column).
                # Then add it onto the voter's pre-existing value (typically 0 for zero-vote countries).
                _norm_val = df.loc[df["FullCountryName"] == 'norm_mean', col].unique()[0]
                _ceil = math.ceil(_norm_val)
                threshold_df.loc[threshold_df["FullCountryName"] == country, col] = _ceil + threshold_df.loc[threshold_df["FullCountryName"] == country, col].unique()[0]
        
        # If audience vote need to be replaced by jury votes
        else :
            print('Votes audience of ', country, ' should be replaced by jury votes of this country.')
            list_replace_audience_by_jury.append(country)
            for col in points_columns:
                threshold_df.loc[threshold_df["FullCountryName"] == country,col] = 0
                
threshold_df["TOTAL_SENT_VOTES"] = threshold_df[votes_columns].sum(axis=1)
threshold_df

Austria  :  audience
Calculating pot votes replacement for  Austria  with no votes delivered process.
Germany  :  audience
Calculating pot votes replacement for  Germany  with no votes delivered process.


United Kingdom  :  audience
Calculating pot votes replacement for  United Kingdom  with no votes delivered process.
Cyprus  :  audience
Calculating pot votes replacement for  Cyprus  with no votes delivered process.


Greece  :  audience
Calculating pot votes replacement for  Greece  with no votes delivered process.
Lithuania  :  audience
Calculating pot votes replacement for  Lithuania  with no votes delivered process.


San Marino  :  audience
Calculating pot votes replacement for  San Marino  with no votes delivered process.


,FullCountryName,strName,POT,TPartnerId,nVotesforDDI01_Australia,nVotesforDDI02_Austria,nVotesforDDI03_Cyprus,nVotesforDDI04_Denmark,nVotesforDDI05_Latvia,nVotesforDDI06_Luxembourg,nVotesforDDI07_Norway,nVotesforDDI08_Romania,nVotesforDDI09_Ukraine,nVotesforDDI10_Czechia,nVotesforDDI11_Switzerland,nVotesforDDI12_France,nVotesforDDI13_United Kingdom,nVotesforDDI14_Belgium,nVotesforDDI15_Estonia,nVotesforDDI16_Finland,nVotesforDDI17_Greece,nVotesforDDI18_Lithuania,nVotesforDDI19_Germany,nVotesforDDI20_Italy,nVotesforDDI21_Poland,nVotesforDDI22_Portugal,nVotesforDDI23_Sweden,nVotesforDDI24_Serbia,nVotesforDDI25_Moldova,TOTAL_SENT_VOTES,ABOVE_POT_THRESHOLD
2,Austria,AT,Pot00,AGGREGATED,40.0,0.0,32.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,57.0,57.0,54.0,36.0,27.0,32.0,64.0,36.0,54.0,52.0,29.0,23.0,25.0,1013.0,False
10,Germany,DE,Pot00,AGGREGATED,40.0,59.0,32.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,57.0,57.0,55.0,36.0,28.0,32.0,0.0,37.0,54.0,52.0,29.0,23.0,25.0,1011.0,False
15,United Kingdom,GB,Pot00,AGGREGATED,40.0,58.0,31.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,0.0,56.0,54.0,36.0,27.0,32.0,64.0,36.0,54.0,52.0,29.0,23.0,25.0,1012.0,False
20,Italy,IT,Pot00,AGGREGATED,202904.0,168401.0,143897.0,155502.0,170648.0,211440.0,194359.0,220953.0,213918.0,156592.0,200775.0,194321.0,156151.0,187433.0,218756.0,215850.0,152135.0,156850.0,201997.0,0.0,144253.0,195237.0,157543.0,130843.0,133062.0,4283820.0,True
14,France,FR,Pot00,AGGREGATED,294776.0,714353.0,264052.0,509591.0,422889.0,463412.0,346891.0,379371.0,402521.0,373523.0,175674.0,0.0,714786.0,639742.0,525482.0,184432.0,166901.0,248685.0,750588.0,340105.0,685252.0,535481.0,182920.0,132997.0,161645.0,9616069.0,True
0,Albania,AL,Pot01,AGGREGATED,2745.0,2975.0,2751.0,2653.0,2864.0,1562.0,1642.0,2286.0,2628.0,2014.0,3315.0,2697.0,3074.0,2761.0,2184.0,3205.0,2063.0,1407.0,1748.0,2028.0,1913.0,1928.0,3230.0,1478.0,1934.0,59085.0,True
6,Bulgaria,BG,Pot01,AGGREGATED,22229.0,19455.0,24175.0,19323.0,22720.0,24252.0,20163.0,17395.0,20718.0,24144.0,21895.0,22950.0,19303.0,19246.0,22443.0,18690.0,18068.0,22314.0,20122.0,24068.0,18582.0,21990.0,24825.0,23107.0,21879.0,534056.0,True
25,Montenegro,ME,Pot01,AGGREGATED,22524.0,22212.0,23962.0,29618.0,25908.0,21388.0,24541.0,22715.0,29399.0,21938.0,29783.0,28602.0,20925.0,24595.0,25072.0,22203.0,26428.0,29374.0,26207.0,28615.0,27955.0,20994.0,26869.0,20455.0,27202.0,629484.0,True
18,Croatia,HR,Pot01,AGGREGATED,84936.0,60041.0,73128.0,71802.0,81572.0,82681.0,69784.0,80594.0,82993.0,75228.0,68185.0,76610.0,79555.0,62539.0,78812.0,71472.0,68956.0,61785.0,69751.0,62473.0,84999.0,60852.0,62894.0,81851.0,59791.0,1813284.0,True
31,Serbia,RS,Pot01,AGGREGATED,55440.0,94232.0,60203.0,87956.0,87672.0,80965.0,77453.0,87240.0,88337.0,60873.0,98133.0,72562.0,86074.0,53033.0,52295.0,97639.0,92251.0,74868.0,54637.0,87670.0,74896.0,72708.0,82940.0,0.0,56239.0,1836316.0,True


---

---

## Stage 4: Jury calculation

Cleaning up the jury dataframe so its columns match the audience matrix (DDI number suffixed with the recipient country name).

In [16]:
############# JURY VOTING 1 #############

# Generating corresponding jury pts 
if selected_df_name == 'sf1_televoting_df':
    jury_df = sf1_jury_df
elif selected_df_name == 'sf2_televoting_df':
    jury_df = sf2_jury_df
elif selected_df_name == 'gf_televoting_df':
    jury_df = gf_jury_df
else:
    raise Exception("Unexpected televoting dataframe name, can't find matching jury dataframe.")

# Append full country names to nVotesforDDIxx columns names
for ddi_key, country_name in corresponding_ddi_dict.items():
    ddi_number = ddi_key[-2:]
    column_to_update = [col for col in jury_df.columns if col.startswith(f"nPointsforDDI{ddi_number}")][0]
    new_column_name = f"Rankfor{ddi_number}_{country_name}"
    jury_df = jury_df.rename(columns={column_to_update: new_column_name})

jury_df

,strName,CountryId,Rankfor01_Australia,Rankfor02_Austria,Rankfor03_Cyprus,Rankfor04_Denmark,Rankfor05_Latvia,Rankfor06_Luxembourg,Rankfor07_Norway,Rankfor08_Romania,Rankfor09_Ukraine,Rankfor10_Czechia,Rankfor11_Switzerland,Rankfor12_France,Rankfor13_United Kingdom,Rankfor14_Belgium,Rankfor15_Estonia,Rankfor16_Finland,Rankfor17_Greece,Rankfor18_Lithuania,Rankfor19_Germany,Rankfor20_Italy,Rankfor21_Poland,Rankfor22_Portugal,Rankfor23_Sweden,Rankfor24_Serbia,Rankfor25_Moldova,nlsReplacementResultFromPotCountries
0,Australia,49,0,13,5,20,8,12,24,21,14,3,2,18,4,16,17,19,7,10,22,6,15,1,23,11,9,0
1,Austria,3,7,0,11,5,19,21,13,20,15,17,2,12,4,18,22,3,24,9,10,6,8,16,23,14,1,0
2,Cyprus,8,20,23,0,9,12,8,17,11,14,4,24,5,2,19,3,1,21,22,15,10,16,6,7,13,18,0
3,Denmark,9,23,13,15,0,5,12,3,2,9,4,11,24,21,18,6,16,14,19,10,1,20,8,17,7,22,0
4,Latvia,19,16,11,17,24,0,10,7,23,8,13,6,15,21,3,4,1,9,5,12,18,14,22,2,19,20,0
5,Luxembourg,53,11,23,9,18,2,0,20,8,14,21,5,19,6,17,7,13,12,10,4,24,1,3,22,16,15,0
6,Norway,24,12,17,6,19,1,9,0,7,10,14,13,11,3,4,24,15,8,2,16,5,18,20,23,21,22,0
7,Romania,27,1,3,7,9,8,22,13,0,16,11,21,18,12,4,15,10,6,17,20,24,2,19,5,23,14,0
8,Ukraine,35,22,4,1,23,10,8,18,12,0,13,15,5,19,7,21,20,11,6,16,2,14,3,9,24,17,0
9,Czechia,41,8,9,4,7,21,2,6,5,20,0,13,23,24,12,3,17,16,14,15,1,11,18,19,10,22,0


---

---

### Stage 4a: Convert jury ranks to Eurovision points

Apply the official mapping (`1 -> 12, 2 -> 10, 3 -> 8, ..., 10 -> 1`) to every `Rankfor##_<Country>` column, producing the corresponding `nPointsforDDI##_<Country>` columns.

In [17]:
############# JURY VOTING 2 #############
# Convert each jury rank into Eurovision points using the official mapping.

jury_pts_df = jury_df


def points_mapping(rank):
    """Map a jury rank (1..10) to Eurovision points; everything else => 0."""
    mapping = {1: 12, 2: 10, 3: 8, 4: 7, 5: 6, 6: 5, 7: 4, 8: 3, 9: 2, 10: 1}
    return mapping.get(rank, 0)


# Add an nPointsforDDI##_<Country> column derived from each Rankfor##_<Country> column
for column in jury_pts_df.columns:
    if column.startswith("Rankfor"):
        new_column_name = column.replace("Rankfor", "nPointsforDDI")
        jury_pts_df[new_column_name] = jury_pts_df[column].apply(points_mapping)

jury_pts_df

,strName,CountryId,Rankfor01_Australia,Rankfor02_Austria,Rankfor03_Cyprus,Rankfor04_Denmark,Rankfor05_Latvia,Rankfor06_Luxembourg,Rankfor07_Norway,Rankfor08_Romania,Rankfor09_Ukraine,Rankfor10_Czechia,Rankfor11_Switzerland,Rankfor12_France,Rankfor13_United Kingdom,Rankfor14_Belgium,Rankfor15_Estonia,Rankfor16_Finland,Rankfor17_Greece,Rankfor18_Lithuania,Rankfor19_Germany,Rankfor20_Italy,Rankfor21_Poland,Rankfor22_Portugal,Rankfor23_Sweden,Rankfor24_Serbia,Rankfor25_Moldova,nlsReplacementResultFromPotCountries,nPointsforDDI01_Australia,nPointsforDDI02_Austria,nPointsforDDI03_Cyprus,nPointsforDDI04_Denmark,nPointsforDDI05_Latvia,nPointsforDDI06_Luxembourg,nPointsforDDI07_Norway,nPointsforDDI08_Romania,nPointsforDDI09_Ukraine,nPointsforDDI10_Czechia,nPointsforDDI11_Switzerland,nPointsforDDI12_France,nPointsforDDI13_United Kingdom,nPointsforDDI14_Belgium,nPointsforDDI15_Estonia,nPointsforDDI16_Finland,nPointsforDDI17_Greece,nPointsforDDI18_Lithuania,nPointsforDDI19_Germany,nPointsforDDI20_Italy,nPointsforDDI21_Poland,nPointsforDDI22_Portugal,nPointsforDDI23_Sweden,nPointsforDDI24_Serbia,nPointsforDDI25_Moldova
0,Australia,49,0,13,5,20,8,12,24,21,14,3,2,18,4,16,17,19,7,10,22,6,15,1,23,11,9,0,0,0,6,0,3,0,0,0,0,8,10,0,7,0,0,0,4,1,0,5,0,12,0,0,2
1,Austria,3,7,0,11,5,19,21,13,20,15,17,2,12,4,18,22,3,24,9,10,6,8,16,23,14,1,0,4,0,0,6,0,0,0,0,0,0,10,0,7,0,0,8,0,2,1,5,3,0,0,0,12
2,Cyprus,8,20,23,0,9,12,8,17,11,14,4,24,5,2,19,3,1,21,22,15,10,16,6,7,13,18,0,0,0,0,2,0,3,0,0,0,7,0,6,10,0,8,12,0,0,0,1,0,5,4,0,0
3,Denmark,9,23,13,15,0,5,12,3,2,9,4,11,24,21,18,6,16,14,19,10,1,20,8,17,7,22,0,0,0,0,0,6,0,8,10,2,7,0,0,0,0,5,0,0,0,1,12,0,3,0,4,0
4,Latvia,19,16,11,17,24,0,10,7,23,8,13,6,15,21,3,4,1,9,5,12,18,14,22,2,19,20,0,0,0,0,0,0,1,4,0,3,0,5,0,0,8,7,12,2,6,0,0,0,0,10,0,0
5,Luxembourg,53,11,23,9,18,2,0,20,8,14,21,5,19,6,17,7,13,12,10,4,24,1,3,22,16,15,0,0,0,2,0,10,0,0,3,0,0,6,0,5,0,4,0,0,1,7,0,12,8,0,0,0
6,Norway,24,12,17,6,19,1,9,0,7,10,14,13,11,3,4,24,15,8,2,16,5,18,20,23,21,22,0,0,0,5,0,12,2,0,4,1,0,0,0,8,7,0,0,3,10,0,6,0,0,0,0,0
7,Romania,27,1,3,7,9,8,22,13,0,16,11,21,18,12,4,15,10,6,17,20,24,2,19,5,23,14,0,12,8,4,2,3,0,0,0,0,0,0,0,0,7,0,1,5,0,0,0,10,0,6,0,0
8,Ukraine,35,22,4,1,23,10,8,18,12,0,13,15,5,19,7,21,20,11,6,16,2,14,3,9,24,17,0,0,7,12,0,1,3,0,0,0,0,0,6,0,4,0,0,0,5,0,10,0,8,2,0,0
9,Czechia,41,8,9,4,7,21,2,6,5,20,0,13,23,24,12,3,17,16,14,15,1,11,18,19,10,22,0,3,2,7,4,0,10,5,6,0,0,0,0,0,0,8,0,0,0,0,12,0,0,0,1,0


---

---

### Stage 4b: Final-jury tie-break (PDF §1.3.1)

Resolve ties on **total jury points** for each recipient country. Per the PDF (§1.3.1, "Tie due to the same number of points from all National Juries"), the chain is:

1. Highest number of national juries that gave it any points.
2. Highest number of `12`-point scores; then `10`, `8`, `7`, `6`, `5`, `4`, `3`, `2`, `1`.
3. Earlier in the show running order wins (replaces the previous alphabetical fallback).

Note: this is **not** the within-jury tie-breaker (majority/youngest/show-of-hands, PDF §1.3.1 second box) — that one is applied upstream by the jury-web interface and is already baked into the per-jury ranks in the input CSV.

In [18]:
############# JURY VOTING 3 : Tie breaking #############

# Step 1: Filter the columns and transpose the filtered DataFrame
filtered_columns = [col for col in jury_pts_df.columns if col.startswith("nPointsforDDI")]
filtered_df = jury_pts_df[filtered_columns]
transposed_df = filtered_df.T

# Step 3: Rename the columns
jury_pts_df = jury_pts_df.rename(columns={"Unnamed: 0": "strName"})
transposed_df.columns = jury_pts_df["strName"].values
# RoW is the Pot 0 jury-pot result, not a real national jury — drop it from the per-country jury aggregation
# (case-insensitive match to handle 'Rest of World' vs 'Rest Of World').
row_cols = [c for c in transposed_df.columns if str(c).lower().replace(' ','') == 'restofworld']
if row_cols:
    transposed_df.drop(columns=row_cols, inplace=True)
jury_tiebreaking_df = transposed_df
country_columns = jury_tiebreaking_df.columns
jury_tiebreaking_df = jury_tiebreaking_df.reset_index()
jury_tiebreaking_df = jury_tiebreaking_df.rename(columns={"index": "Recipient_Country"})


jury_tiebreaking_df["Pts_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row != 0).sum(), axis=1)
jury_tiebreaking_df["12_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row == 12).sum(), axis=1)
jury_tiebreaking_df["10_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row == 10).sum(), axis=1)
jury_tiebreaking_df["8_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row == 8).sum(), axis=1)
jury_tiebreaking_df["7_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row == 7).sum(), axis=1)
jury_tiebreaking_df["6_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row == 6).sum(), axis=1)
jury_tiebreaking_df["5_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row == 5).sum(), axis=1)
jury_tiebreaking_df["4_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row == 4).sum(), axis=1)
jury_tiebreaking_df["3_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row == 3).sum(), axis=1)
jury_tiebreaking_df["2_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row == 2).sum(), axis=1)
jury_tiebreaking_df["1_from_X_National_Juries"] = jury_tiebreaking_df[country_columns].apply(lambda row: (row == 1).sum(), axis=1)

# Sort in tiebreaking order
# Final tie-break per EBU PDF §1.3.1: jury count → 12-pt count → 10-pt count → … → 1-pt count
# → show running order (earlier wins). The DDI prefix in 'Recipient_Country' encodes the show order.
jury_tiebreaking_df["_ShowOrder"] = (
    jury_tiebreaking_df["Recipient_Country"].str.extract(r"DDI(\d+)")[0].astype(int)
)
jury_tiebreaking_df = jury_tiebreaking_df.sort_values(
    by=[
        "Pts_from_X_National_Juries",
        "12_from_X_National_Juries",
        "10_from_X_National_Juries",
        "8_from_X_National_Juries",
        "7_from_X_National_Juries",
        "6_from_X_National_Juries",
        "5_from_X_National_Juries",
        "4_from_X_National_Juries",
        "3_from_X_National_Juries",
        "2_from_X_National_Juries",
        "1_from_X_National_Juries",
        "_ShowOrder",
    ],
    ascending=[False, False, False, False, False, False, False, False, False, False, False, True],
)
jury_tiebreaking_df = jury_tiebreaking_df.drop(columns=["_ShowOrder"])

columns = [
    "Pts_from_X_National_Juries",
    "12_from_X_National_Juries",
    "10_from_X_National_Juries",
    "8_from_X_National_Juries",
    "7_from_X_National_Juries",
    "6_from_X_National_Juries",
    "5_from_X_National_Juries",
    "4_from_X_National_Juries",
    "3_from_X_National_Juries",
    "2_from_X_National_Juries",
    "1_from_X_National_Juries",
]

# Move the specified columns right after the first column
cols = jury_tiebreaking_df.columns.tolist()
for col in reversed(columns):
    if col in cols:
        cols.insert(1, cols.pop(cols.index(col)))
jury_tiebreaking_df = jury_tiebreaking_df[cols]

# Add a new column 'TIEBREAKING_RANK'
jury_tiebreaking_df.insert(1, "TIEBREAKING_RANK", range(1, len(jury_tiebreaking_df) + 1))

# Save the new dataframe
jury_tiebreaking_df.to_excel(os.path.join(SCRIPT_XLSX_DIRECTORY, "03.jury_received_pts_tiebreakers.xlsx"), index=False)

jury_tiebreaking_df

,Recipient_Country,TIEBREAKING_RANK,Pts_from_X_National_Juries,12_from_X_National_Juries,10_from_X_National_Juries,8_from_X_National_Juries,7_from_X_National_Juries,6_from_X_National_Juries,5_from_X_National_Juries,4_from_X_National_Juries,3_from_X_National_Juries,2_from_X_National_Juries,1_from_X_National_Juries,Australia,Austria,Cyprus,Denmark,Latvia,Luxembourg,Norway,Romania,Ukraine,Czechia,Switzerland,France,United Kingdom,Belgium,Estonia,Finland,Greece,Lithuania,Germany,Italy,Poland,Portugal,Sweden,Serbia,Moldova,Albania,Armenia,Azerbaijan,Bulgaria,Croatia,Georgia,Israel,Malta,Montenegro,San Marino
4,nPointsforDDI05_Latvia,1,22,7,2,1,0,1,3,1,5,0,2,3,0,0,6,0,10,12,3,1,0,3,8,0,3,3,12,5,0,12,5,0,12,12,10,0,0,0,12,5,12,0,4,1,0,0
0,nPointsforDDI01_Australia,2,21,1,3,4,2,2,0,2,4,2,1,0,4,0,0,0,0,0,12,0,3,0,1,0,0,0,4,7,0,8,6,8,8,3,2,10,3,2,8,6,0,3,0,10,10,7
21,nPointsforDDI22_Portugal,3,19,2,2,3,3,1,1,3,1,1,2,12,0,5,3,0,8,0,0,8,0,7,7,0,0,0,0,0,12,0,10,4,0,1,0,6,0,0,10,7,1,4,2,8,4,0
19,nPointsforDDI20_Italy,4,17,3,1,0,0,4,3,0,2,0,4,5,5,1,12,0,0,6,0,10,12,0,0,1,0,6,0,0,6,0,0,1,6,0,3,0,5,0,0,0,0,0,1,3,12,0
1,nPointsforDDI02_Austria,5,16,2,1,3,2,0,1,0,3,3,1,0,0,0,0,0,0,0,8,7,2,1,0,3,8,7,0,12,2,0,0,0,0,2,0,0,10,0,0,3,0,8,12,0,5,3
14,nPointsforDDI15_Estonia,6,16,1,0,2,2,3,4,1,1,0,2,0,0,8,5,7,4,0,0,0,8,6,0,0,0,0,0,6,0,3,1,0,0,0,0,5,7,6,5,12,0,0,0,5,0,1
6,nPointsforDDI07_Norway,7,16,0,1,4,1,0,2,5,0,3,0,0,0,0,8,4,0,0,0,0,5,8,0,8,7,0,0,0,4,5,4,0,0,0,4,8,0,0,0,2,2,10,0,4,0,2
2,nPointsforDDI03_Cyprus,8,15,5,0,0,3,1,3,1,1,1,0,6,0,0,0,0,2,5,4,12,7,12,5,12,0,0,0,0,0,0,0,7,0,5,0,12,0,12,3,0,0,0,7,0,0,0
22,nPointsforDDI23_Sweden,9,15,1,2,1,1,1,1,2,1,3,2,0,0,4,0,10,0,0,6,2,0,0,0,0,0,1,0,0,7,10,3,5,0,0,1,0,4,8,2,0,0,0,0,0,2,12
8,nPointsforDDI09_Ukraine,10,15,0,3,1,2,0,0,1,4,3,1,0,0,0,2,3,0,1,0,0,0,10,2,4,0,0,10,0,3,0,7,3,0,7,0,0,0,0,0,0,8,2,10,0,3,0


---

---

### Stage 4c: Jury total points

Add two summary rows to the jury matrix:

- `Total_with_RoW` - sum of jury points received from every national jury (incl. Rest Of World).
- `Total_without_RoW` - same total, but excluding Rest Of World.

Also add `nDifferenceforDDI*` columns derived from the rank, used later as a tie-break helper inside the televoting stage.

In [19]:
############# JURY VOTING 3 : Total points #############
# Add Total_with_RoW and Total_without_RoW summary rows, plus the nDifferenceforDDI*
# helper columns used later as a tie-breaker on the televoting side.

jury_totals_df = jury_pts_df
pts_columns = [col for col in jury_totals_df.columns if col.startswith("nPointsforDDI")]

# --- Row: Total_with_RoW (sum of all national juries, including Rest Of World) ---
new_row_sums_jury = pd.DataFrame(columns=jury_totals_df.columns, index=[len(jury_totals_df)])
for col in pts_columns:
    new_row_sums_jury[col] = jury_totals_df[col].sum()
new_row_sums_jury["strName"] = "Total_with_RoW"
jury_totals_df = pd.concat([jury_totals_df, new_row_sums_jury], ignore_index=True)

# --- Row: Total_without_RoW (Total_with_RoW minus the Rest Of World contribution) ---
new_row_without_RoW = pd.DataFrame(columns=jury_totals_df.columns, index=[len(jury_totals_df)])
new_row_without_RoW["strName"] = "Total_without_RoW"
# Match RoW case-insensitively (CSV may use "Rest of World" or "Rest Of World")
row_RoW = jury_totals_df[jury_totals_df["strName"].str.lower().str.replace(" ","") == "restofworld"]

for col in pts_columns:
    column_sum_without_RoW = (
        jury_totals_df.loc[jury_totals_df["strName"] == "Total_with_RoW", col].values[0]
        - row_RoW[col].values[0]
    )
    new_row_without_RoW[col] = column_sum_without_RoW

jury_totals_df = pd.concat([jury_totals_df, new_row_without_RoW], ignore_index=True)

# Note: removed the nDifferenceforDDI* helper columns (the "1 - rank/10000" perturbation hack).
# The audience tie-break per PDF §1.2.1 — "song with best rank from the National Jury wins" —
# is now applied cleanly in the televoting cells using the actual jury rank as the tie-break key.

jury_totals_df

,strName,CountryId,Rankfor01_Australia,Rankfor02_Austria,Rankfor03_Cyprus,Rankfor04_Denmark,Rankfor05_Latvia,Rankfor06_Luxembourg,Rankfor07_Norway,Rankfor08_Romania,Rankfor09_Ukraine,Rankfor10_Czechia,Rankfor11_Switzerland,Rankfor12_France,Rankfor13_United Kingdom,Rankfor14_Belgium,Rankfor15_Estonia,Rankfor16_Finland,Rankfor17_Greece,Rankfor18_Lithuania,Rankfor19_Germany,Rankfor20_Italy,Rankfor21_Poland,Rankfor22_Portugal,Rankfor23_Sweden,Rankfor24_Serbia,Rankfor25_Moldova,nlsReplacementResultFromPotCountries,nPointsforDDI01_Australia,nPointsforDDI02_Austria,nPointsforDDI03_Cyprus,nPointsforDDI04_Denmark,nPointsforDDI05_Latvia,nPointsforDDI06_Luxembourg,nPointsforDDI07_Norway,nPointsforDDI08_Romania,nPointsforDDI09_Ukraine,nPointsforDDI10_Czechia,nPointsforDDI11_Switzerland,nPointsforDDI12_France,nPointsforDDI13_United Kingdom,nPointsforDDI14_Belgium,nPointsforDDI15_Estonia,nPointsforDDI16_Finland,nPointsforDDI17_Greece,nPointsforDDI18_Lithuania,nPointsforDDI19_Germany,nPointsforDDI20_Italy,nPointsforDDI21_Poland,nPointsforDDI22_Portugal,nPointsforDDI23_Sweden,nPointsforDDI24_Serbia,nPointsforDDI25_Moldova
0,Australia,49,0,13,5,20,8,12,24,21,14,3,2,18,4,16,17,19,7,10,22,6,15,1,23,11,9,0,0,0,6,0,3,0,0,0,0,8,10,0,7,0,0,0,4,1,0,5,0,12,0,0,2
1,Austria,3,7,0,11,5,19,21,13,20,15,17,2,12,4,18,22,3,24,9,10,6,8,16,23,14,1,0,4,0,0,6,0,0,0,0,0,0,10,0,7,0,0,8,0,2,1,5,3,0,0,0,12
2,Cyprus,8,20,23,0,9,12,8,17,11,14,4,24,5,2,19,3,1,21,22,15,10,16,6,7,13,18,0,0,0,0,2,0,3,0,0,0,7,0,6,10,0,8,12,0,0,0,1,0,5,4,0,0
3,Denmark,9,23,13,15,0,5,12,3,2,9,4,11,24,21,18,6,16,14,19,10,1,20,8,17,7,22,0,0,0,0,0,6,0,8,10,2,7,0,0,0,0,5,0,0,0,1,12,0,3,0,4,0
4,Latvia,19,16,11,17,24,0,10,7,23,8,13,6,15,21,3,4,1,9,5,12,18,14,22,2,19,20,0,0,0,0,0,0,1,4,0,3,0,5,0,0,8,7,12,2,6,0,0,0,0,10,0,0
5,Luxembourg,53,11,23,9,18,2,0,20,8,14,21,5,19,6,17,7,13,12,10,4,24,1,3,22,16,15,0,0,0,2,0,10,0,0,3,0,0,6,0,5,0,4,0,0,1,7,0,12,8,0,0,0
6,Norway,24,12,17,6,19,1,9,0,7,10,14,13,11,3,4,24,15,8,2,16,5,18,20,23,21,22,0,0,0,5,0,12,2,0,4,1,0,0,0,8,7,0,0,3,10,0,6,0,0,0,0,0
7,Romania,27,1,3,7,9,8,22,13,0,16,11,21,18,12,4,15,10,6,17,20,24,2,19,5,23,14,0,12,8,4,2,3,0,0,0,0,0,0,0,0,7,0,1,5,0,0,0,10,0,6,0,0
8,Ukraine,35,22,4,1,23,10,8,18,12,0,13,15,5,19,7,21,20,11,6,16,2,14,3,9,24,17,0,0,7,12,0,1,3,0,0,0,0,0,6,0,4,0,0,0,5,0,10,0,8,2,0,0
9,Czechia,41,8,9,4,7,21,2,6,5,20,0,13,23,24,12,3,17,16,14,15,1,11,18,19,10,22,0,3,2,7,4,0,10,5,6,0,0,0,0,0,0,8,0,0,0,0,12,0,0,0,1,0


---

---

### Stage 4d: Jury global rankings

Rank every recipient country based on `Total_with_RoW` and `Total_without_RoW`, producing two new rows (`Rank_with_RoW`, `Rank_without_RoW`).

In [20]:
############# JURY VOTING 4 : CALCULATE RANK #############
# Add Rank_with_RoW and Rank_without_RoW rows derived from the totals rows above.

jury_global_rankings_df = jury_totals_df


def _build_rank_row(label, totals_label):
    """Return a one-row dataframe holding the per-country jury rank derived from ``totals_label``."""
    row = pd.DataFrame(columns=jury_global_rankings_df.columns, index=[len(jury_global_rankings_df)])
    row["strName"] = label
    totals = jury_global_rankings_df.loc[
        jury_global_rankings_df["strName"] == totals_label, pts_columns
    ]
    ranks = totals.rank(axis=1, ascending=False, method="min").astype(int)
    for i, col in enumerate(pts_columns):
        rank_col_name = col.replace("nPointsforDDI", "Rankfor")
        row[rank_col_name] = ranks.iloc[0, i]
    return row


jury_global_rankings_df = pd.concat(
    [jury_global_rankings_df, _build_rank_row("Rank_with_RoW", "Total_with_RoW")],
    ignore_index=True,
)
jury_global_rankings_df = pd.concat(
    [jury_global_rankings_df, _build_rank_row("Rank_without_RoW", "Total_without_RoW")],
    ignore_index=True,
)

jury_global_rankings_df

,strName,CountryId,Rankfor01_Australia,Rankfor02_Austria,Rankfor03_Cyprus,Rankfor04_Denmark,Rankfor05_Latvia,Rankfor06_Luxembourg,Rankfor07_Norway,Rankfor08_Romania,Rankfor09_Ukraine,Rankfor10_Czechia,Rankfor11_Switzerland,Rankfor12_France,Rankfor13_United Kingdom,Rankfor14_Belgium,Rankfor15_Estonia,Rankfor16_Finland,Rankfor17_Greece,Rankfor18_Lithuania,Rankfor19_Germany,Rankfor20_Italy,Rankfor21_Poland,Rankfor22_Portugal,Rankfor23_Sweden,Rankfor24_Serbia,Rankfor25_Moldova,nlsReplacementResultFromPotCountries,nPointsforDDI01_Australia,nPointsforDDI02_Austria,nPointsforDDI03_Cyprus,nPointsforDDI04_Denmark,nPointsforDDI05_Latvia,nPointsforDDI06_Luxembourg,nPointsforDDI07_Norway,nPointsforDDI08_Romania,nPointsforDDI09_Ukraine,nPointsforDDI10_Czechia,nPointsforDDI11_Switzerland,nPointsforDDI12_France,nPointsforDDI13_United Kingdom,nPointsforDDI14_Belgium,nPointsforDDI15_Estonia,nPointsforDDI16_Finland,nPointsforDDI17_Greece,nPointsforDDI18_Lithuania,nPointsforDDI19_Germany,nPointsforDDI20_Italy,nPointsforDDI21_Poland,nPointsforDDI22_Portugal,nPointsforDDI23_Sweden,nPointsforDDI24_Serbia,nPointsforDDI25_Moldova
0,Australia,49,0,13,5,20,8,12,24,21,14,3,2,18,4,16,17,19,7,10,22,6,15,1,23,11,9,0,0,0,6,0,3,0,0,0,0,8,10,0,7,0,0,0,4,1,0,5,0,12,0,0,2
1,Austria,3,7,0,11,5,19,21,13,20,15,17,2,12,4,18,22,3,24,9,10,6,8,16,23,14,1,0,4,0,0,6,0,0,0,0,0,0,10,0,7,0,0,8,0,2,1,5,3,0,0,0,12
2,Cyprus,8,20,23,0,9,12,8,17,11,14,4,24,5,2,19,3,1,21,22,15,10,16,6,7,13,18,0,0,0,0,2,0,3,0,0,0,7,0,6,10,0,8,12,0,0,0,1,0,5,4,0,0
3,Denmark,9,23,13,15,0,5,12,3,2,9,4,11,24,21,18,6,16,14,19,10,1,20,8,17,7,22,0,0,0,0,0,6,0,8,10,2,7,0,0,0,0,5,0,0,0,1,12,0,3,0,4,0
4,Latvia,19,16,11,17,24,0,10,7,23,8,13,6,15,21,3,4,1,9,5,12,18,14,22,2,19,20,0,0,0,0,0,0,1,4,0,3,0,5,0,0,8,7,12,2,6,0,0,0,0,10,0,0
5,Luxembourg,53,11,23,9,18,2,0,20,8,14,21,5,19,6,17,7,13,12,10,4,24,1,3,22,16,15,0,0,0,2,0,10,0,0,3,0,0,6,0,5,0,4,0,0,1,7,0,12,8,0,0,0
6,Norway,24,12,17,6,19,1,9,0,7,10,14,13,11,3,4,24,15,8,2,16,5,18,20,23,21,22,0,0,0,5,0,12,2,0,4,1,0,0,0,8,7,0,0,3,10,0,6,0,0,0,0,0
7,Romania,27,1,3,7,9,8,22,13,0,16,11,21,18,12,4,15,10,6,17,20,24,2,19,5,23,14,0,12,8,4,2,3,0,0,0,0,0,0,0,0,7,0,1,5,0,0,0,10,0,6,0,0
8,Ukraine,35,22,4,1,23,10,8,18,12,0,13,15,5,19,7,21,20,11,6,16,2,14,3,9,24,17,0,0,7,12,0,1,3,0,0,0,0,0,6,0,4,0,0,0,5,0,10,0,8,2,0,0
9,Czechia,41,8,9,4,7,21,2,6,5,20,0,13,23,24,12,3,17,16,14,15,1,11,18,19,10,22,0,3,2,7,4,0,10,5,6,0,0,0,0,0,0,8,0,0,0,0,12,0,0,0,1,0


---

---

### Stage 4e: Final jury results

Transpose the jury rankings so each row is a recipient country, attach the tie-breaking rank, sort using `(Total_without_RoW DESC, TIEBREAKING_RANK ASC)` and assign the final 1..N jury rank.

In [21]:
############# JURY FINAL RESULTS #############

# Build JURY final dataframe 
final_jury_results_df = jury_global_rankings_df.set_index('strName')
transposed_final_jury_results_df = final_jury_results_df.T
transposed_final_jury_results_df = transposed_final_jury_results_df.reset_index()
transposed_final_jury_results_df = transposed_final_jury_results_df.rename(columns={'index': 'Recipient_Country'})

# Filter rows that correspond to the 'nVotesforDDI' columns
transposed_final_jury_results_df = transposed_final_jury_results_df.sort_values(by='Total_without_RoW', ascending=False)

# Replace the RoW Jury with 0


# Move the 'jury_PTS_TOTAL' column to the second position
cols = transposed_final_jury_results_df.columns.tolist()
cols.insert(1, cols.pop(cols.index('Total_with_RoW')))
cols.insert(1, cols.pop(cols.index('Total_without_RoW')))
transposed_final_jury_results_df = transposed_final_jury_results_df[cols]

transposed_final_jury_results_w_tiebreakers_df = transposed_final_jury_results_df
jury_tiebreaking_ranks_only_df = jury_tiebreaking_df[['Recipient_Country', 'TIEBREAKING_RANK']]
transposed_final_jury_results_w_tiebreakers_df = transposed_final_jury_results_w_tiebreakers_df.merge(jury_tiebreaking_ranks_only_df, on='Recipient_Country', how='left')

# Move the 'jury_PTS_TOTAL' column to the second position
cols = transposed_final_jury_results_w_tiebreakers_df.columns.tolist()
cols.insert(2, cols.pop(cols.index('TIEBREAKING_RANK')))
transposed_final_jury_results_w_tiebreakers_df = transposed_final_jury_results_w_tiebreakers_df[cols]

# Sort for final order
transposed_final_jury_results_w_tiebreakers_df = transposed_final_jury_results_w_tiebreakers_df.sort_values(by=['Total_without_RoW', 'TIEBREAKING_RANK'], ascending=[False, True])
transposed_final_jury_results_w_tiebreakers_df.insert(0, 'Final_Rank', range(1, len(transposed_final_jury_results_w_tiebreakers_df) + 1))

# save
transposed_final_jury_results_w_tiebreakers_df.to_excel(os.path.join(SCRIPT_XLSX_DIRECTORY, "04.jury_final_jury_results_w_tiebreaker.xlsx"), index=False)

transposed_final_jury_results_w_tiebreakers_df

,Final_Rank,Recipient_Country,Total_without_RoW,TIEBREAKING_RANK,Total_with_RoW,Australia,Austria,Cyprus,Denmark,Latvia,Luxembourg,Norway,Romania,Ukraine,Czechia,Switzerland,France,United Kingdom,Belgium,Estonia,Finland,Greece,Lithuania,Germany,Italy,Poland,Portugal,Sweden,Serbia,Moldova,Albania,Armenia,Azerbaijan,Bulgaria,Croatia,Georgia,Israel,Malta,Montenegro,San Marino,Rest Of World,Rank_with_RoW,Rank_without_RoW
0,1,nPointsforDDI05_Latvia,154,1.0,166,3,0,0,6,0,10,12,3,1,0,3,8,0,3,3,12,5,0,12,5,0,12,12,10,0,0,0,12,5,12,0,4,1,0,0,12,NaN,NaN
1,2,nPointsforDDI01_Australia,125,2.0,131,0,4,0,0,0,0,0,12,0,3,0,1,0,0,0,4,7,0,8,6,8,8,3,2,10,3,2,8,6,0,3,0,10,10,7,6,NaN,NaN
2,3,nPointsforDDI22_Portugal,119,3.0,122,12,0,5,3,0,8,0,0,8,0,7,7,0,0,0,0,0,12,0,10,4,0,1,0,6,0,0,10,7,1,4,2,8,4,0,3,NaN,NaN
3,4,nPointsforDDI03_Cyprus,111,8.0,116,6,0,0,0,0,2,5,4,12,7,12,5,12,0,0,0,0,0,0,0,7,0,5,0,12,0,12,3,0,0,0,7,0,0,0,5,NaN,NaN
4,5,nPointsforDDI14_Belgium,101,14.0,101,0,0,0,0,8,0,7,7,4,0,0,0,5,0,0,0,8,0,0,0,0,0,10,6,0,12,0,0,10,4,12,0,0,0,8,0,NaN,NaN
5,6,nPointsforDDI20_Italy,95,4.0,95,5,5,1,12,0,0,6,0,10,12,0,0,1,0,6,0,0,6,0,0,1,6,0,3,0,5,0,0,0,0,0,1,3,12,0,0,NaN,NaN
6,7,nPointsforDDI02_Austria,93,5.0,93,0,0,0,0,0,0,0,8,7,2,1,0,3,8,7,0,12,2,0,0,0,0,2,0,0,10,0,0,3,0,8,12,0,5,3,0,NaN,NaN
7,8,nPointsforDDI16_Finland,90,12.0,92,0,8,12,0,12,0,0,1,0,0,0,6,0,12,0,0,3,0,0,0,12,2,4,8,0,0,5,1,4,0,0,0,0,0,0,2,NaN,NaN
8,9,nPointsforDDI15_Estonia,89,6.0,89,0,0,8,5,7,4,0,0,0,8,6,0,0,0,0,0,6,0,3,1,0,0,0,0,5,7,6,5,12,0,0,0,5,0,1,0,NaN,NaN
9,10,nPointsforDDI21_Poland,87,16.0,87,0,3,0,0,0,12,0,10,0,0,5,10,0,10,10,6,0,0,0,0,0,1,0,0,0,0,0,7,0,0,7,0,6,0,0,0,NaN,NaN


---

## Stage 4f: Replace audience voting by jury for selected countries

For any country flagged in `list_replace_audience_by_jury` (typically San Marino, or any country that fell back to jury during pot calculation), copy the jury points into the audience matrix so the final televoting calculation uses the jury values instead.

In [22]:
def replace_audience_w_jury(country, df_jury, df_audience):
    pts_columns_audiences = [col for col in df_audience.columns if col.startswith("nVotesforDDI")]
    pts_columns_jury = [col for col in df_jury.columns if col.startswith("nPointsforDDI")]
    for col_audience in pts_columns_audiences:
        for col_jury in pts_columns_jury:
            if col_jury.split('_')[1] == col_audience.split('_')[1]:
                df_audience.loc[df_audience["FullCountryName"]==country, col_audience] = df_jury.loc[df_jury["strName"]==country, col_jury].unique()[0]
    return df_audience

### Inspect the list of countries to replace with jury votes


In [23]:
#list_replace_audience_by_jury.append('San Marino')
list_replace_audience_by_jury

[]

### Apply jury replacement on the audience matrix


In [24]:
print('Replacing audience voting by jury voting for : ' , list_replace_audience_by_jury )
for country in list_replace_audience_by_jury:
    threshold_df = replace_audience_w_jury(country, jury_global_rankings_df, threshold_df)
threshold_df

Replacing audience voting by jury voting for :  []


,FullCountryName,strName,POT,TPartnerId,nVotesforDDI01_Australia,nVotesforDDI02_Austria,nVotesforDDI03_Cyprus,nVotesforDDI04_Denmark,nVotesforDDI05_Latvia,nVotesforDDI06_Luxembourg,nVotesforDDI07_Norway,nVotesforDDI08_Romania,nVotesforDDI09_Ukraine,nVotesforDDI10_Czechia,nVotesforDDI11_Switzerland,nVotesforDDI12_France,nVotesforDDI13_United Kingdom,nVotesforDDI14_Belgium,nVotesforDDI15_Estonia,nVotesforDDI16_Finland,nVotesforDDI17_Greece,nVotesforDDI18_Lithuania,nVotesforDDI19_Germany,nVotesforDDI20_Italy,nVotesforDDI21_Poland,nVotesforDDI22_Portugal,nVotesforDDI23_Sweden,nVotesforDDI24_Serbia,nVotesforDDI25_Moldova,TOTAL_SENT_VOTES,ABOVE_POT_THRESHOLD
2,Austria,AT,Pot00,AGGREGATED,40.0,0.0,32.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,57.0,57.0,54.0,36.0,27.0,32.0,64.0,36.0,54.0,52.0,29.0,23.0,25.0,1013.0,False
10,Germany,DE,Pot00,AGGREGATED,40.0,59.0,32.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,57.0,57.0,55.0,36.0,28.0,32.0,0.0,37.0,54.0,52.0,29.0,23.0,25.0,1011.0,False
15,United Kingdom,GB,Pot00,AGGREGATED,40.0,58.0,31.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,0.0,56.0,54.0,36.0,27.0,32.0,64.0,36.0,54.0,52.0,29.0,23.0,25.0,1012.0,False
20,Italy,IT,Pot00,AGGREGATED,202904.0,168401.0,143897.0,155502.0,170648.0,211440.0,194359.0,220953.0,213918.0,156592.0,200775.0,194321.0,156151.0,187433.0,218756.0,215850.0,152135.0,156850.0,201997.0,0.0,144253.0,195237.0,157543.0,130843.0,133062.0,4283820.0,True
14,France,FR,Pot00,AGGREGATED,294776.0,714353.0,264052.0,509591.0,422889.0,463412.0,346891.0,379371.0,402521.0,373523.0,175674.0,0.0,714786.0,639742.0,525482.0,184432.0,166901.0,248685.0,750588.0,340105.0,685252.0,535481.0,182920.0,132997.0,161645.0,9616069.0,True
0,Albania,AL,Pot01,AGGREGATED,2745.0,2975.0,2751.0,2653.0,2864.0,1562.0,1642.0,2286.0,2628.0,2014.0,3315.0,2697.0,3074.0,2761.0,2184.0,3205.0,2063.0,1407.0,1748.0,2028.0,1913.0,1928.0,3230.0,1478.0,1934.0,59085.0,True
6,Bulgaria,BG,Pot01,AGGREGATED,22229.0,19455.0,24175.0,19323.0,22720.0,24252.0,20163.0,17395.0,20718.0,24144.0,21895.0,22950.0,19303.0,19246.0,22443.0,18690.0,18068.0,22314.0,20122.0,24068.0,18582.0,21990.0,24825.0,23107.0,21879.0,534056.0,True
25,Montenegro,ME,Pot01,AGGREGATED,22524.0,22212.0,23962.0,29618.0,25908.0,21388.0,24541.0,22715.0,29399.0,21938.0,29783.0,28602.0,20925.0,24595.0,25072.0,22203.0,26428.0,29374.0,26207.0,28615.0,27955.0,20994.0,26869.0,20455.0,27202.0,629484.0,True
18,Croatia,HR,Pot01,AGGREGATED,84936.0,60041.0,73128.0,71802.0,81572.0,82681.0,69784.0,80594.0,82993.0,75228.0,68185.0,76610.0,79555.0,62539.0,78812.0,71472.0,68956.0,61785.0,69751.0,62473.0,84999.0,60852.0,62894.0,81851.0,59791.0,1813284.0,True
31,Serbia,RS,Pot01,AGGREGATED,55440.0,94232.0,60203.0,87956.0,87672.0,80965.0,77453.0,87240.0,88337.0,60873.0,98133.0,72562.0,86074.0,53033.0,52295.0,97639.0,92251.0,74868.0,54637.0,87670.0,74896.0,72708.0,82940.0,0.0,56239.0,1836316.0,True


---

## Stage 5: Televoting calculation

Merge the audience matrix with the jury `nDifferenceforDDI*` helper columns. Those small (< 1) values will be added to the audience votes as a deterministic tie-breaker before ranking.

In [25]:
############# TELEVOTING 1 #############
# Per PDF §1.2.1: when two songs receive the same number of audience votes in a given
# country, the tie is broken by that country's own National Jury rank (lower = better).
# We attach each voting country's jury Rankfor* columns to its televoting row so we can
# break ties cleanly when ranking audience votes below.

if required_pot_calculation:
    televoting_final_votes_w_jury_pts_df = threshold_df
else:
    televoting_final_votes_w_jury_pts_df = pot_df

# Pull each country's own jury final ranks as tie-break keys
jury_rank_cols = [c for c in jury_global_rankings_df.columns if c.startswith("Rankfor")]
jury_ranks_for_tiebreak = jury_global_rankings_df[["strName"] + jury_rank_cols].copy()

televoting_final_votes_w_jury_pts_df = televoting_final_votes_w_jury_pts_df.merge(
    jury_ranks_for_tiebreak, left_on="FullCountryName", right_on="strName", how="left"
)

televoting_final_votes_w_jury_pts_df

,FullCountryName,strName_x,POT,TPartnerId,nVotesforDDI01_Australia,nVotesforDDI02_Austria,nVotesforDDI03_Cyprus,nVotesforDDI04_Denmark,nVotesforDDI05_Latvia,nVotesforDDI06_Luxembourg,nVotesforDDI07_Norway,nVotesforDDI08_Romania,nVotesforDDI09_Ukraine,nVotesforDDI10_Czechia,nVotesforDDI11_Switzerland,nVotesforDDI12_France,nVotesforDDI13_United Kingdom,nVotesforDDI14_Belgium,nVotesforDDI15_Estonia,nVotesforDDI16_Finland,nVotesforDDI17_Greece,nVotesforDDI18_Lithuania,nVotesforDDI19_Germany,nVotesforDDI20_Italy,nVotesforDDI21_Poland,nVotesforDDI22_Portugal,nVotesforDDI23_Sweden,nVotesforDDI24_Serbia,nVotesforDDI25_Moldova,TOTAL_SENT_VOTES,ABOVE_POT_THRESHOLD,strName_y,Rankfor01_Australia,Rankfor02_Austria,Rankfor03_Cyprus,Rankfor04_Denmark,Rankfor05_Latvia,Rankfor06_Luxembourg,Rankfor07_Norway,Rankfor08_Romania,Rankfor09_Ukraine,Rankfor10_Czechia,Rankfor11_Switzerland,Rankfor12_France,Rankfor13_United Kingdom,Rankfor14_Belgium,Rankfor15_Estonia,Rankfor16_Finland,Rankfor17_Greece,Rankfor18_Lithuania,Rankfor19_Germany,Rankfor20_Italy,Rankfor21_Poland,Rankfor22_Portugal,Rankfor23_Sweden,Rankfor24_Serbia,Rankfor25_Moldova
0,Austria,AT,Pot00,AGGREGATED,40.0,0.0,32.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,57.0,57.0,54.0,36.0,27.0,32.0,64.0,36.0,54.0,52.0,29.0,23.0,25.0,1013.0,False,Austria,7,0,11,5,19,21,13,20,15,17,2,12,4,18,22,3,24,9,10,6,8,16,23,14,1
1,Germany,DE,Pot00,AGGREGATED,40.0,59.0,32.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,57.0,57.0,55.0,36.0,28.0,32.0,0.0,37.0,54.0,52.0,29.0,23.0,25.0,1011.0,False,Germany,3,20,13,21,1,23,6,24,14,5,9,4,17,18,8,19,7,10,0,15,16,22,2,11,12
2,United Kingdom,GB,Pot00,AGGREGATED,40.0,58.0,31.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,0.0,56.0,54.0,36.0,27.0,32.0,64.0,36.0,54.0,52.0,29.0,23.0,25.0,1012.0,False,United Kingdom,16,8,1,20,11,15,3,2,7,9,14,24,0,6,12,18,23,21,4,10,22,19,17,5,13
3,Italy,IT,Pot00,AGGREGATED,202904.0,168401.0,143897.0,155502.0,170648.0,211440.0,194359.0,220953.0,213918.0,156592.0,200775.0,194321.0,156151.0,187433.0,218756.0,215850.0,152135.0,156850.0,201997.0,0.0,144253.0,195237.0,157543.0,130843.0,133062.0,4283820.0,True,Italy,5,20,19,9,6,15,7,11,4,14,23,1,18,12,10,13,21,16,22,0,24,2,8,17,3
4,France,FR,Pot00,AGGREGATED,294776.0,714353.0,264052.0,509591.0,422889.0,463412.0,346891.0,379371.0,402521.0,373523.0,175674.0,0.0,714786.0,639742.0,525482.0,184432.0,166901.0,248685.0,750588.0,340105.0,685252.0,535481.0,182920.0,132997.0,161645.0,9616069.0,True,France,10,24,6,11,3,18,21,12,9,20,17,0,13,14,16,5,7,15,19,22,2,4,23,1,8
5,Albania,AL,Pot01,AGGREGATED,2745.0,2975.0,2751.0,2653.0,2864.0,1562.0,1642.0,2286.0,2628.0,2014.0,3315.0,2697.0,3074.0,2761.0,2184.0,3205.0,2063.0,1407.0,1748.0,2028.0,1913.0,1928.0,3230.0,1478.0,1934.0,59085.0,True,Albania,8,2,13,19,16,18,23,10,17,11,15,14,3,1,4,21,20,5,22,6,12,25,7,24,9
6,Bulgaria,BG,Pot01,AGGREGATED,22229.0,19455.0,24175.0,19323.0,22720.0,24252.0,20163.0,17395.0,20718.0,24144.0,21895.0,22950.0,19303.0,19246.0,22443.0,18690.0,18068.0,22314.0,20122.0,24068.0,18582.0,21990.0,24825.0,23107.0,21879.0,534056.0,True,Bulgaria,5,8,17,10,6,23,9,13,19,20,3,14,15,2,1,7,11,21,16,25,24,4,12,22,18
7,Montenegro,ME,Pot01,AGGREGATED,22524.0,22212.0,23962.0,29618.0,25908.0,21388.0,24541.0,22715.0,29399.0,21938.0,29783.0,28602.0,20925.0,24595.0,25072.0,22203.0,26428.0,29374.0,26207.0,28615.0,27955.0,20994.0,26869.0,20455.0,27202.0,629484.0,True,Montenegro,2,6,24,5,17,14,18,22,8,16,13,11,20,23,15,19,4,12,10,1,21,7,9,3,25
8,Croatia,HR,Pot01,AGGREGATED,84936.0,60041.0,73128.0,71802.0,81572.0,82681.0,69784.0,80594.0,82993.0,75228.0,68185.0,76610.0,79555.0,62539.0,78812.0,71472.0,68956.0,61785.0,69751.0,62473.0,84999.0,60852.0,62894.0,81851.0,59791.0,1813284.0,True,Croatia,18,16,20,22,1,6,9,17,3,2,23,13,21,7,12,11,5,25,19,24,14,10,15,4,8
9,Serbia,RS,Pot01,AGGREGATED,55440.0,94232.0,60203.0,87956.0,87672.0,80965.0,77453.0,87240.0,88337.0,60873.0,98133.0,72562.0,86074.0,53033.0,52295.0,97639.0,92251.0,74868.0,54637.0,87670.0,74896.0,72708.0,8294

---

### Stage 5b: Audience tie-breaking

Avoid ties in the audience ranking by adding the jury-derived difference (`nDifferenceforDDI* / 100`) to each `nVotesforDDI*` column before ranking.

### Quick inspection of the merged televoting + jury-difference matrix


In [26]:
televoting_final_votes_w_jury_pts_df

,FullCountryName,strName_x,POT,TPartnerId,nVotesforDDI01_Australia,nVotesforDDI02_Austria,nVotesforDDI03_Cyprus,nVotesforDDI04_Denmark,nVotesforDDI05_Latvia,nVotesforDDI06_Luxembourg,nVotesforDDI07_Norway,nVotesforDDI08_Romania,nVotesforDDI09_Ukraine,nVotesforDDI10_Czechia,nVotesforDDI11_Switzerland,nVotesforDDI12_France,nVotesforDDI13_United Kingdom,nVotesforDDI14_Belgium,nVotesforDDI15_Estonia,nVotesforDDI16_Finland,nVotesforDDI17_Greece,nVotesforDDI18_Lithuania,nVotesforDDI19_Germany,nVotesforDDI20_Italy,nVotesforDDI21_Poland,nVotesforDDI22_Portugal,nVotesforDDI23_Sweden,nVotesforDDI24_Serbia,nVotesforDDI25_Moldova,TOTAL_SENT_VOTES,ABOVE_POT_THRESHOLD,strName_y,Rankfor01_Australia,Rankfor02_Austria,Rankfor03_Cyprus,Rankfor04_Denmark,Rankfor05_Latvia,Rankfor06_Luxembourg,Rankfor07_Norway,Rankfor08_Romania,Rankfor09_Ukraine,Rankfor10_Czechia,Rankfor11_Switzerland,Rankfor12_France,Rankfor13_United Kingdom,Rankfor14_Belgium,Rankfor15_Estonia,Rankfor16_Finland,Rankfor17_Greece,Rankfor18_Lithuania,Rankfor19_Germany,Rankfor20_Italy,Rankfor21_Poland,Rankfor22_Portugal,Rankfor23_Sweden,Rankfor24_Serbia,Rankfor25_Moldova
0,Austria,AT,Pot00,AGGREGATED,40.0,0.0,32.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,57.0,57.0,54.0,36.0,27.0,32.0,64.0,36.0,54.0,52.0,29.0,23.0,25.0,1013.0,False,Austria,7,0,11,5,19,21,13,20,15,17,2,12,4,18,22,3,24,9,10,6,8,16,23,14,1
1,Germany,DE,Pot00,AGGREGATED,40.0,59.0,32.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,57.0,57.0,55.0,36.0,28.0,32.0,0.0,37.0,54.0,52.0,29.0,23.0,25.0,1011.0,False,Germany,3,20,13,21,1,23,6,24,14,5,9,4,17,18,8,19,7,10,0,15,16,22,2,11,12
2,United Kingdom,GB,Pot00,AGGREGATED,40.0,58.0,31.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,0.0,56.0,54.0,36.0,27.0,32.0,64.0,36.0,54.0,52.0,29.0,23.0,25.0,1012.0,False,United Kingdom,16,8,1,20,11,15,3,2,7,9,14,24,0,6,12,18,23,21,4,10,22,19,17,5,13
3,Italy,IT,Pot00,AGGREGATED,202904.0,168401.0,143897.0,155502.0,170648.0,211440.0,194359.0,220953.0,213918.0,156592.0,200775.0,194321.0,156151.0,187433.0,218756.0,215850.0,152135.0,156850.0,201997.0,0.0,144253.0,195237.0,157543.0,130843.0,133062.0,4283820.0,True,Italy,5,20,19,9,6,15,7,11,4,14,23,1,18,12,10,13,21,16,22,0,24,2,8,17,3
4,France,FR,Pot00,AGGREGATED,294776.0,714353.0,264052.0,509591.0,422889.0,463412.0,346891.0,379371.0,402521.0,373523.0,175674.0,0.0,714786.0,639742.0,525482.0,184432.0,166901.0,248685.0,750588.0,340105.0,685252.0,535481.0,182920.0,132997.0,161645.0,9616069.0,True,France,10,24,6,11,3,18,21,12,9,20,17,0,13,14,16,5,7,15,19,22,2,4,23,1,8
5,Albania,AL,Pot01,AGGREGATED,2745.0,2975.0,2751.0,2653.0,2864.0,1562.0,1642.0,2286.0,2628.0,2014.0,3315.0,2697.0,3074.0,2761.0,2184.0,3205.0,2063.0,1407.0,1748.0,2028.0,1913.0,1928.0,3230.0,1478.0,1934.0,59085.0,True,Albania,8,2,13,19,16,18,23,10,17,11,15,14,3,1,4,21,20,5,22,6,12,25,7,24,9
6,Bulgaria,BG,Pot01,AGGREGATED,22229.0,19455.0,24175.0,19323.0,22720.0,24252.0,20163.0,17395.0,20718.0,24144.0,21895.0,22950.0,19303.0,19246.0,22443.0,18690.0,18068.0,22314.0,20122.0,24068.0,18582.0,21990.0,24825.0,23107.0,21879.0,534056.0,True,Bulgaria,5,8,17,10,6,23,9,13,19,20,3,14,15,2,1,7,11,21,16,25,24,4,12,22,18
7,Montenegro,ME,Pot01,AGGREGATED,22524.0,22212.0,23962.0,29618.0,25908.0,21388.0,24541.0,22715.0,29399.0,21938.0,29783.0,28602.0,20925.0,24595.0,25072.0,22203.0,26428.0,29374.0,26207.0,28615.0,27955.0,20994.0,26869.0,20455.0,27202.0,629484.0,True,Montenegro,2,6,24,5,17,14,18,22,8,16,13,11,20,23,15,19,4,12,10,1,21,7,9,3,25
8,Croatia,HR,Pot01,AGGREGATED,84936.0,60041.0,73128.0,71802.0,81572.0,82681.0,69784.0,80594.0,82993.0,75228.0,68185.0,76610.0,79555.0,62539.0,78812.0,71472.0,68956.0,61785.0,69751.0,62473.0,84999.0,60852.0,62894.0,81851.0,59791.0,1813284.0,True,Croatia,18,16,20,22,1,6,9,17,3,2,23,13,21,7,12,11,5,25,19,24,14,10,15,4,8
9,Serbia,RS,Pot01,AGGREGATED,55440.0,94232.0,60203.0,87956.0,87672.0,80965.0,77453.0,87240.0,88337.0,60873.0,98133.0,72562.0,86074.0,53033.0,52295.0,97639.0,92251.0,74868.0,54637.0,87670.0,74896.0,72708.0,8294

### Stage 5c: Apply jury-derived tie-breaker to the audience votes


In [27]:
############# TELEVOTING 2 #############
# Previously this cell perturbed audience vote counts by tiny fractions derived from the
# jury rank (the nDifferenceforDDI hack). That implicit tie-break is replaced by an
# explicit ranking step in TELEVOTING 3 that uses the jury rank as a secondary sort key.
# This cell is now a no-op kept for compatibility with downstream cell numbering.

televoting_final_votes_w_jury_pts_df

,FullCountryName,strName_x,POT,TPartnerId,nVotesforDDI01_Australia,nVotesforDDI02_Austria,nVotesforDDI03_Cyprus,nVotesforDDI04_Denmark,nVotesforDDI05_Latvia,nVotesforDDI06_Luxembourg,nVotesforDDI07_Norway,nVotesforDDI08_Romania,nVotesforDDI09_Ukraine,nVotesforDDI10_Czechia,nVotesforDDI11_Switzerland,nVotesforDDI12_France,nVotesforDDI13_United Kingdom,nVotesforDDI14_Belgium,nVotesforDDI15_Estonia,nVotesforDDI16_Finland,nVotesforDDI17_Greece,nVotesforDDI18_Lithuania,nVotesforDDI19_Germany,nVotesforDDI20_Italy,nVotesforDDI21_Poland,nVotesforDDI22_Portugal,nVotesforDDI23_Sweden,nVotesforDDI24_Serbia,nVotesforDDI25_Moldova,TOTAL_SENT_VOTES,ABOVE_POT_THRESHOLD,strName_y,Rankfor01_Australia,Rankfor02_Austria,Rankfor03_Cyprus,Rankfor04_Denmark,Rankfor05_Latvia,Rankfor06_Luxembourg,Rankfor07_Norway,Rankfor08_Romania,Rankfor09_Ukraine,Rankfor10_Czechia,Rankfor11_Switzerland,Rankfor12_France,Rankfor13_United Kingdom,Rankfor14_Belgium,Rankfor15_Estonia,Rankfor16_Finland,Rankfor17_Greece,Rankfor18_Lithuania,Rankfor19_Germany,Rankfor20_Italy,Rankfor21_Poland,Rankfor22_Portugal,Rankfor23_Sweden,Rankfor24_Serbia,Rankfor25_Moldova
0,Austria,AT,Pot00,AGGREGATED,40.0,0.0,32.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,57.0,57.0,54.0,36.0,27.0,32.0,64.0,36.0,54.0,52.0,29.0,23.0,25.0,1013.0,False,Austria,7,0,11,5,19,21,13,20,15,17,2,12,4,18,22,3,24,9,10,6,8,16,23,14,1
1,Germany,DE,Pot00,AGGREGATED,40.0,59.0,32.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,57.0,57.0,55.0,36.0,28.0,32.0,0.0,37.0,54.0,52.0,29.0,23.0,25.0,1011.0,False,Germany,3,20,13,21,1,23,6,24,14,5,9,4,17,18,8,19,7,10,0,15,16,22,2,11,12
2,United Kingdom,GB,Pot00,AGGREGATED,40.0,58.0,31.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,0.0,56.0,54.0,36.0,27.0,32.0,64.0,36.0,54.0,52.0,29.0,23.0,25.0,1012.0,False,United Kingdom,16,8,1,20,11,15,3,2,7,9,14,24,0,6,12,18,23,21,4,10,22,19,17,5,13
3,Italy,IT,Pot00,AGGREGATED,202904.0,168401.0,143897.0,155502.0,170648.0,211440.0,194359.0,220953.0,213918.0,156592.0,200775.0,194321.0,156151.0,187433.0,218756.0,215850.0,152135.0,156850.0,201997.0,0.0,144253.0,195237.0,157543.0,130843.0,133062.0,4283820.0,True,Italy,5,20,19,9,6,15,7,11,4,14,23,1,18,12,10,13,21,16,22,0,24,2,8,17,3
4,France,FR,Pot00,AGGREGATED,294776.0,714353.0,264052.0,509591.0,422889.0,463412.0,346891.0,379371.0,402521.0,373523.0,175674.0,0.0,714786.0,639742.0,525482.0,184432.0,166901.0,248685.0,750588.0,340105.0,685252.0,535481.0,182920.0,132997.0,161645.0,9616069.0,True,France,10,24,6,11,3,18,21,12,9,20,17,0,13,14,16,5,7,15,19,22,2,4,23,1,8
5,Albania,AL,Pot01,AGGREGATED,2745.0,2975.0,2751.0,2653.0,2864.0,1562.0,1642.0,2286.0,2628.0,2014.0,3315.0,2697.0,3074.0,2761.0,2184.0,3205.0,2063.0,1407.0,1748.0,2028.0,1913.0,1928.0,3230.0,1478.0,1934.0,59085.0,True,Albania,8,2,13,19,16,18,23,10,17,11,15,14,3,1,4,21,20,5,22,6,12,25,7,24,9
6,Bulgaria,BG,Pot01,AGGREGATED,22229.0,19455.0,24175.0,19323.0,22720.0,24252.0,20163.0,17395.0,20718.0,24144.0,21895.0,22950.0,19303.0,19246.0,22443.0,18690.0,18068.0,22314.0,20122.0,24068.0,18582.0,21990.0,24825.0,23107.0,21879.0,534056.0,True,Bulgaria,5,8,17,10,6,23,9,13,19,20,3,14,15,2,1,7,11,21,16,25,24,4,12,22,18
7,Montenegro,ME,Pot01,AGGREGATED,22524.0,22212.0,23962.0,29618.0,25908.0,21388.0,24541.0,22715.0,29399.0,21938.0,29783.0,28602.0,20925.0,24595.0,25072.0,22203.0,26428.0,29374.0,26207.0,28615.0,27955.0,20994.0,26869.0,20455.0,27202.0,629484.0,True,Montenegro,2,6,24,5,17,14,18,22,8,16,13,11,20,23,15,19,4,12,10,1,21,7,9,3,25
8,Croatia,HR,Pot01,AGGREGATED,84936.0,60041.0,73128.0,71802.0,81572.0,82681.0,69784.0,80594.0,82993.0,75228.0,68185.0,76610.0,79555.0,62539.0,78812.0,71472.0,68956.0,61785.0,69751.0,62473.0,84999.0,60852.0,62894.0,81851.0,59791.0,1813284.0,True,Croatia,18,16,20,22,1,6,9,17,3,2,23,13,21,7,12,11,5,25,19,24,14,10,15,4,8
9,Serbia,RS,Pot01,AGGREGATED,55440.0,94232.0,60203.0,87956.0,87672.0,80965.0,77453.0,87240.0,88337.0,60873.0,98133.0,72562.0,86074.0,53033.0,52295.0,97639.0,92251.0,74868.0,54637.0,87670.0,74896.0,72708.0,8294

### Stage 5d: Rank audience votes per voting country


In [28]:
############# TELEVOTING 3 #############
# Rank audience votes per voting country, breaking ties by the country's own jury rank
# (lower jury rank = better) per PDF §1.2.1.

import numpy as np

votes_columns = [col for col in televoting_final_votes_w_jury_pts_df.columns if col.startswith('nVotesforDDI')]
rank_columns  = [col.replace('nVotesforDDI', 'RankforDDI') for col in votes_columns]

def _rank_with_jury_tiebreak(row):
    # For each DDI column: (-votes, jury_rank). lexsort gives small-first so negating votes
    # makes higher-vote = better. jury_rank is naturally smaller-is-better.
    keys = []
    for v_col in votes_columns:
        ddi = v_col.replace('nVotesforDDI', '')      # e.g. '01_Moldova'
        jury_col = 'Rankfor' + ddi                   # matches the merged column name
        votes = row[v_col]
        jury_r = row.get(jury_col, np.nan)
        if pd.isna(jury_r) or jury_r == 0:
            jury_r = 99   # disqualified / home / unknown — push to the back of any tie
        keys.append((-votes, jury_r))
    # Compute final 1..N order
    order = sorted(range(len(keys)), key=lambda i: keys[i])
    ranks = [0] * len(keys)
    for pos, i in enumerate(order):
        ranks[i] = pos + 1
    return pd.Series(ranks, index=rank_columns)

televoting_final_votes_w_jury_pts_df[rank_columns] = (
    televoting_final_votes_w_jury_pts_df.apply(_rank_with_jury_tiebreak, axis=1)
)

televoting_final_votes_w_jury_pts_df

,FullCountryName,strName_x,POT,TPartnerId,nVotesforDDI01_Australia,nVotesforDDI02_Austria,nVotesforDDI03_Cyprus,nVotesforDDI04_Denmark,nVotesforDDI05_Latvia,nVotesforDDI06_Luxembourg,nVotesforDDI07_Norway,nVotesforDDI08_Romania,nVotesforDDI09_Ukraine,nVotesforDDI10_Czechia,nVotesforDDI11_Switzerland,nVotesforDDI12_France,nVotesforDDI13_United Kingdom,nVotesforDDI14_Belgium,nVotesforDDI15_Estonia,nVotesforDDI16_Finland,nVotesforDDI17_Greece,nVotesforDDI18_Lithuania,nVotesforDDI19_Germany,nVotesforDDI20_Italy,nVotesforDDI21_Poland,nVotesforDDI22_Portugal,nVotesforDDI23_Sweden,nVotesforDDI24_Serbia,nVotesforDDI25_Moldova,TOTAL_SENT_VOTES,ABOVE_POT_THRESHOLD,strName_y,Rankfor01_Australia,Rankfor02_Austria,Rankfor03_Cyprus,Rankfor04_Denmark,Rankfor05_Latvia,Rankfor06_Luxembourg,Rankfor07_Norway,Rankfor08_Romania,Rankfor09_Ukraine,Rankfor10_Czechia,Rankfor11_Switzerland,Rankfor12_France,Rankfor13_United Kingdom,Rankfor14_Belgium,Rankfor15_Estonia,Rankfor16_Finland,Rankfor17_Greece,Rankfor18_Lithuania,Rankfor19_Germany,Rankfor20_Italy,Rankfor21_Poland,Rankfor22_Portugal,Rankfor23_Sweden,Rankfor24_Serbia,Rankfor25_Moldova,RankforDDI01_Australia,RankforDDI02_Austria,RankforDDI03_Cyprus,RankforDDI04_Denmark,RankforDDI05_Latvia,RankforDDI06_Luxembourg,RankforDDI07_Norway,RankforDDI08_Romania,RankforDDI09_Ukraine,RankforDDI10_Czechia,RankforDDI11_Switzerland,RankforDDI12_France,RankforDDI13_United Kingdom,RankforDDI14_Belgium,RankforDDI15_Estonia,RankforDDI16_Finland,RankforDDI17_Greece,RankforDDI18_Lithuania,RankforDDI19_Germany,RankforDDI20_Italy,RankforDDI21_Poland,RankforDDI22_Portugal,RankforDDI23_Sweden,RankforDDI24_Serbia,RankforDDI25_Moldova
0,Austria,AT,Pot00,AGGREGATED,40.0,0.0,32.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,57.0,57.0,54.0,36.0,27.0,32.0,64.0,36.0,54.0,52.0,29.0,23.0,25.0,1013.0,False,Austria,7,0,11,5,19,21,13,20,15,17,2,12,4,18,22,3,24,9,10,6,8,16,23,14,1,14,25,20,11,12,7,13,10,9,15,18,8,2,3,5,16,22,19,1,17,4,6,21,24,23
1,Germany,DE,Pot00,AGGREGATED,40.0,59.0,32.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,57.0,57.0,55.0,36.0,28.0,32.0,0.0,37.0,54.0,52.0,29.0,23.0,25.0,1011.0,False,Germany,3,20,13,21,1,23,6,24,14,5,9,4,17,18,8,19,7,10,0,15,16,22,2,11,12,14,1,20,11,12,7,13,10,9,15,18,8,2,3,4,17,22,19,25,16,5,6,21,24,23
2,United Kingdom,GB,Pot00,AGGREGATED,40.0,58.0,31.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,0.0,56.0,54.0,36.0,27.0,32.0,64.0,36.0,54.0,52.0,29.0,23.0,25.0,1012.0,False,United Kingdom,16,8,1,20,11,15,3,2,7,9,14,24,0,6,12,18,23,21,4,10,22,19,17,5,13,14,2,20,11,12,7,13,8,9,15,18,10,25,3,4,17,22,19,1,16,5,6,21,24,23
3,Italy,IT,Pot00,AGGREGATED,202904.0,168401.0,143897.0,155502.0,170648.0,211440.0,194359.0,220953.0,213918.0,156592.0,200775.0,194321.0,156151.0,187433.0,218756.0,215850.0,152135.0,156850.0,201997.0,0.0,144253.0,195237.0,157543.0,130843.0,133062.0,4283820.0,True,Italy,5,20,19,9,6,15,7,11,4,14,23,1,18,12,10,13,21,16,22,0,24,2,8,17,3,6,14,22,19,13,5,10,1,4,17,8,11,18,12,2,3,20,16,7,25,21,9,15,24,23
4,France,FR,Pot00,AGGREGATED,294776.0,714353.0,264052.0,509591.0,422889.0,463412.0,346891.0,379371.0,402521.0,373523.0,175674.0,0.0,714786.0,639742.0,525482.0,184432.0,166901.0,248685.0,750588.0,340105.0,685252.0,535481.0,182920.0,132997.0,161645.0,9616069.0,True,France,10,24,6,11,3,18,21,12,9,20,17,0,13,14,16,5,7,15,19,22,2,4,23,1,8,16,3,17,8,10,9,14,12,11,13,21,25,2,5,7,19,22,18,1,15,4,6,20,24,23
5,Albania,AL,Pot01,AGGREGATED,2745.0,2975.0,2751.0,2653.0,2864.0,1562.0,1642.0,2286.0,2628.0,2014.0,3315.0,2697.0,3074.0,2761.0,2184.0,3205.0,2063.0,1407.0,1748.0,2028.0,1913.0,1928.0,3230.0,1478.0,1934.0,59085.0,True,Albania,8,2,13,19,16,18,23,10,17,11,15,14,3,1,4,21,20,5,22,6,12,25,7,24,9,9,5,8,11,6,23,22,13,12,17,1,10,4,7,14,3,15,25,21,16,20,19,2,24,18
6,Bulgaria,BG,Pot01,AGGREGATED,22229.0,19455.0,24175.0,19323.0,22720.0,24252.0,20163.0,17395.0,20718.0,24144.0,21895.0,22950.0,19303.0,19246.0,22443.0,18690.0,18068.0,22314.0,20122.0,24068.0,18582.0,21990.0,24825.0,23107.0,21879.0,534056.0,Tr

### Stage 5e: Map audience ranks to Eurovision points


In [29]:
############# TELEVOTING 4 #############

# Change ranks to points
televoting_final_pts_df = televoting_final_votes_w_jury_pts_df
rank_columns = [col for col in televoting_final_pts_df.columns if col.startswith('RankforDDI')]

mapping = {1: 12, 2: 10, 3: 8, 4: 7, 5: 6, 6: 5, 7: 4, 8: 3, 9: 2, 10: 1}

for col in rank_columns:
    points_col = col.replace('RankforDDI','nPointsforDDI')
    televoting_final_pts_df[points_col] = televoting_final_pts_df[col].apply(lambda x: mapping.get(x, 0) if x <= 10 else 0)
    
televoting_final_pts_df

,FullCountryName,strName_x,POT,TPartnerId,nVotesforDDI01_Australia,nVotesforDDI02_Austria,nVotesforDDI03_Cyprus,nVotesforDDI04_Denmark,nVotesforDDI05_Latvia,nVotesforDDI06_Luxembourg,nVotesforDDI07_Norway,nVotesforDDI08_Romania,nVotesforDDI09_Ukraine,nVotesforDDI10_Czechia,nVotesforDDI11_Switzerland,nVotesforDDI12_France,nVotesforDDI13_United Kingdom,nVotesforDDI14_Belgium,nVotesforDDI15_Estonia,nVotesforDDI16_Finland,nVotesforDDI17_Greece,nVotesforDDI18_Lithuania,nVotesforDDI19_Germany,nVotesforDDI20_Italy,nVotesforDDI21_Poland,nVotesforDDI22_Portugal,nVotesforDDI23_Sweden,nVotesforDDI24_Serbia,nVotesforDDI25_Moldova,TOTAL_SENT_VOTES,ABOVE_POT_THRESHOLD,strName_y,Rankfor01_Australia,Rankfor02_Austria,Rankfor03_Cyprus,Rankfor04_Denmark,Rankfor05_Latvia,Rankfor06_Luxembourg,Rankfor07_Norway,Rankfor08_Romania,Rankfor09_Ukraine,Rankfor10_Czechia,Rankfor11_Switzerland,Rankfor12_France,Rankfor13_United Kingdom,Rankfor14_Belgium,Rankfor15_Estonia,Rankfor16_Finland,Rankfor17_Greece,Rankfor18_Lithuania,Rankfor19_Germany,Rankfor20_Italy,Rankfor21_Poland,Rankfor22_Portugal,Rankfor23_Sweden,Rankfor24_Serbia,Rankfor25_Moldova,RankforDDI01_Australia,RankforDDI02_Austria,RankforDDI03_Cyprus,RankforDDI04_Denmark,RankforDDI05_Latvia,RankforDDI06_Luxembourg,RankforDDI07_Norway,RankforDDI08_Romania,RankforDDI09_Ukraine,RankforDDI10_Czechia,RankforDDI11_Switzerland,RankforDDI12_France,RankforDDI13_United Kingdom,RankforDDI14_Belgium,RankforDDI15_Estonia,RankforDDI16_Finland,RankforDDI17_Greece,RankforDDI18_Lithuania,RankforDDI19_Germany,RankforDDI20_Italy,RankforDDI21_Poland,RankforDDI22_Portugal,RankforDDI23_Sweden,RankforDDI24_Serbia,RankforDDI25_Moldova,nPointsforDDI01_Australia,nPointsforDDI02_Austria,nPointsforDDI03_Cyprus,nPointsforDDI04_Denmark,nPointsforDDI05_Latvia,nPointsforDDI06_Luxembourg,nPointsforDDI07_Norway,nPointsforDDI08_Romania,nPointsforDDI09_Ukraine,nPointsforDDI10_Czechia,nPointsforDDI11_Switzerland,nPointsforDDI12_France,nPointsforDDI13_United Kingdom,nPointsforDDI14_Belgium,nPointsforDDI15_Estonia,nPointsforDDI16_Finland,nPointsforDDI17_Greece,nPointsforDDI18_Lithuania,nPointsforDDI19_Germany,nPointsforDDI20_Italy,nPointsforDDI21_Poland,nPointsforDDI22_Portugal,nPointsforDDI23_Sweden,nPointsforDDI24_Serbia,nPointsforDDI25_Moldova
0,Austria,AT,Pot00,AGGREGATED,40.0,0.0,32.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,57.0,57.0,54.0,36.0,27.0,32.0,64.0,36.0,54.0,52.0,29.0,23.0,25.0,1013.0,False,Austria,7,0,11,5,19,21,13,20,15,17,2,12,4,18,22,3,24,9,10,6,8,16,23,14,1,14,25,20,11,12,7,13,10,9,15,18,8,2,3,5,16,22,19,1,17,4,6,21,24,23,0,0,0,0,0,4,0,1,2,0,0,3,10,8,6,0,0,0,12,0,7,5,0,0,0
1,Germany,DE,Pot00,AGGREGATED,40.0,59.0,32.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,57.0,57.0,55.0,36.0,28.0,32.0,0.0,37.0,54.0,52.0,29.0,23.0,25.0,1011.0,False,Germany,3,20,13,21,1,23,6,24,14,5,9,4,17,18,8,19,7,10,0,15,16,22,2,11,12,14,1,20,11,12,7,13,10,9,15,18,8,2,3,4,17,22,19,25,16,5,6,21,24,23,0,12,0,0,0,4,0,1,2,0,0,3,10,8,7,0,0,0,0,0,6,5,0,0,0
2,United Kingdom,GB,Pot00,AGGREGATED,40.0,58.0,31.0,46.0,43.0,50.0,42.0,47.0,47.0,39.0,34.0,47.0,0.0,56.0,54.0,36.0,27.0,32.0,64.0,36.0,54.0,52.0,29.0,23.0,25.0,1012.0,False,United Kingdom,16,8,1,20,11,15,3,2,7,9,14,24,0,6,12,18,23,21,4,10,22,19,17,5,13,14,2,20,11,12,7,13,8,9,15,18,10,25,3,4,17,22,19,1,16,5,6,21,24,23,0,10,0,0,0,4,0,3,2,0,0,1,0,8,7,0,0,0,12,0,6,5,0,0,0
3,Italy,IT,Pot00,AGGREGATED,202904.0,168401.0,143897.0,155502.0,170648.0,211440.0,194359.0,220953.0,213918.0,156592.0,200775.0,194321.0,156151.0,187433.0,218756.0,215850.0,152135.0,156850.0,201997.0,0.0,144253.0,195237.0,157543.0,130843.0,133062.0,4283820.0,True,Italy,5,20,19,9,6,15,7,11,4,14,23,1,18,12,10,13,21,16,22,0,24,2,8,17,3,6,14,22,19,13,5,10,1,4,17,8,11,18,12,2,3,20,16,7,25,21,9,15,24,23,5,0,0,0,0,6,1,12,7,0,3,0,0,0,10,8,0,0,4,0,0,2,0,0,0
4,France,FR,Pot00,AGGREGATED,294776.0,714353.0,264052.0,509591.0,422889.0,463412.0,346891.0,379371.0,402521.0,373523.0,175674.0,0.0,714786.0,639742.0,525482.0,184432.0,166901.0,248685.0

### Stage 5f: Compute televoting tie-breaking rank


In [30]:
############# TELEVOTING 5 : Tie break #############

# Step 1: Filter the columns
filtered_columns = [col for col in televoting_final_pts_df.columns if col.startswith("nPointsforDDI")]
filtered_df = televoting_final_pts_df[filtered_columns]

transposed_df = filtered_df.T
transposed_df.columns = televoting_final_pts_df["FullCountryName"].values
tv_tiebreaking_df = transposed_df
tv_tiebreaking_df = tv_tiebreaking_df.rename(columns={"index": "Recipient_Country"})

country_columns = tv_tiebreaking_df.columns

tv_tiebreaking_df["Pts_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row != 0).sum(), axis=1)
tv_tiebreaking_df["12_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row == 12).sum(), axis=1)
tv_tiebreaking_df["10_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row == 10).sum(), axis=1)
tv_tiebreaking_df["8_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row == 8).sum(), axis=1)
tv_tiebreaking_df["7_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row == 7).sum(), axis=1)
tv_tiebreaking_df["6_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row == 6).sum(), axis=1)
tv_tiebreaking_df["5_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row == 5).sum(), axis=1)
tv_tiebreaking_df["4_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row == 4).sum(), axis=1)
tv_tiebreaking_df["3_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row == 3).sum(), axis=1)
tv_tiebreaking_df["2_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row == 2).sum(), axis=1)
tv_tiebreaking_df["1_from_X_National_Audiences"] = tv_tiebreaking_df[country_columns].apply(lambda row: (row == 1).sum(), axis=1)

tv_tiebreaking_df = tv_tiebreaking_df.reset_index()
tv_tiebreaking_df = tv_tiebreaking_df.rename(columns={"index": "Country"})

# Final tie-break per EBU PDF §1.2.1: audience count → 12-pt count → 10-pt count → … → 1-pt count
# → show running order (earlier wins). The DDI prefix in 'Country' encodes the show order.
tv_tiebreaking_df["_ShowOrder"] = (
    tv_tiebreaking_df["Country"].str.extract(r"DDI(\d+)")[0].astype(int)
)
tv_tiebreaking_df = tv_tiebreaking_df.sort_values(
    by=[
        "Pts_from_X_National_Audiences",
        "12_from_X_National_Audiences",
        "10_from_X_National_Audiences",
        "8_from_X_National_Audiences",
        "7_from_X_National_Audiences",
        "6_from_X_National_Audiences",
        "5_from_X_National_Audiences",
        "4_from_X_National_Audiences",
        "3_from_X_National_Audiences",
        "2_from_X_National_Audiences",
        "1_from_X_National_Audiences",
        "_ShowOrder",
    ],
    ascending=[False, False, False, False, False, False, False, False, False, False, False, True],
)
tv_tiebreaking_df = tv_tiebreaking_df.drop(columns=["_ShowOrder"])

columns = [
    "Pts_from_X_National_Audiences",
    "12_from_X_National_Audiences",
    "10_from_X_National_Audiences",
    "8_from_X_National_Audiences",
    "7_from_X_National_Audiences",
    "6_from_X_National_Audiences",
    "6_from_X_National_Audiences",
    "5_from_X_National_Audiences",
    "4_from_X_National_Audiences",
    "3_from_X_National_Audiences",
    "2_from_X_National_Audiences",
    "1_from_X_National_Audiences",
]

# Move the specified columns right after the first column
cols = tv_tiebreaking_df.columns.tolist()
for col in reversed(columns):
    if col in cols:
        cols.insert(1, cols.pop(cols.index(col)))
tv_tiebreaking_df = tv_tiebreaking_df[cols]

# Add a new column 'TIEBREAKING_RANK'
tv_tiebreaking_df.insert(1, "TIEBREAKING_RANK", range(1, len(tv_tiebreaking_df) + 1))
tv_tiebreaking_df.to_excel(os.path.join(SCRIPT_XLSX_DIRECTORY, "05.televoting_received_pts_tiebreakers.xlsx"), index=False)

tv_tiebreaking_df

,Country,TIEBREAKING_RANK,Pts_from_X_National_Audiences,12_from_X_National_Audiences,10_from_X_National_Audiences,8_from_X_National_Audiences,7_from_X_National_Audiences,6_from_X_National_Audiences,5_from_X_National_Audiences,4_from_X_National_Audiences,3_from_X_National_Audiences,2_from_X_National_Audiences,1_from_X_National_Audiences,Austria,Germany,United Kingdom,Italy,France,Albania,Bulgaria,Montenegro,Croatia,Serbia,Switzerland,Estonia,Finland,Australia,Denmark,Norway,Sweden,Georgia,Azerbaijan,Armenia,Israel,Poland,Ukraine,Moldova,Romania,Luxembourg,Portugal,Czechia,Belgium,Cyprus,Greece,Lithuania,San Marino,Malta,Latvia,Rest Of World
18,nPointsforDDI19_Germany,1,21,4,3,3,1,3,0,2,0,2,3,12,0,12,4,12,0,0,0,0,0,7,0,0,6,10,10,6,8,0,0,12,0,10,0,6,1,0,4,8,2,2,1,1,8,0,0
20,nPointsforDDI21_Poland,2,20,2,2,5,2,3,1,1,1,2,1,7,6,6,0,7,0,0,4,12,0,0,1,0,0,8,0,2,0,0,5,0,0,6,10,2,8,0,0,0,12,8,8,8,3,10,0
14,nPointsforDDI15_Estonia,3,20,0,2,3,4,4,0,3,0,3,1,6,7,7,10,4,0,2,0,2,0,10,0,0,0,0,0,4,2,0,1,6,4,0,0,8,0,8,0,0,7,7,6,6,0,8,0
21,nPointsforDDI22_Portugal,4,19,4,1,1,1,1,5,1,3,1,1,5,5,5,2,5,0,0,0,0,0,12,12,0,3,3,7,10,4,12,6,0,5,1,0,0,12,0,8,0,0,0,0,0,0,0,3
13,nPointsforDDI14_Belgium,5,19,3,3,3,4,1,1,1,0,0,3,8,8,8,0,6,4,0,0,0,0,0,0,0,0,7,1,0,1,0,7,0,0,0,1,12,0,10,7,0,10,10,12,12,5,7,0
5,nPointsforDDI06_Luxembourg,6,17,3,3,1,1,2,0,4,1,2,0,4,4,4,6,2,0,10,0,7,0,8,10,0,0,12,12,3,0,0,0,10,6,12,4,0,0,0,2,0,0,0,0,0,0,0,0
8,nPointsforDDI09_Ukraine,7,17,0,2,5,1,2,1,1,1,4,0,2,2,2,7,0,0,0,8,8,6,6,2,8,4,0,8,0,0,0,0,3,0,0,5,0,0,0,0,10,0,0,0,0,10,0,8
11,nPointsforDDI12_France,8,17,0,2,1,0,2,2,3,2,1,4,3,3,1,0,0,1,4,5,1,0,4,0,4,0,2,0,0,0,0,10,5,8,0,6,0,6,0,1,0,0,0,0,0,0,0,10
2,nPointsforDDI03_Cyprus,9,15,2,2,1,4,1,0,1,1,2,1,0,0,0,0,0,3,8,0,0,0,2,0,7,7,0,0,7,0,0,0,7,0,0,2,0,0,4,0,1,0,12,10,10,6,12,0
1,nPointsforDDI02_Austria,10,15,1,4,2,1,1,1,1,3,0,1,0,12,10,0,8,6,0,0,0,8,0,0,0,0,0,0,0,10,10,3,0,0,0,0,10,0,1,0,7,3,5,3,4,0,0,0


### Stage 5g: Add the `TELEVOTING_PTS_TOTAL` row


In [31]:
############# TELEVOTING 5 : Tie break #############
# Add a TELEVOTING_PTS_TOTAL row summing nPointsforDDI* across every voting country.

votes_columns = [col for col in televoting_final_pts_df.columns if col.startswith("nPointsforDDI")]
new_row_sums_votes = pd.DataFrame(columns=televoting_final_pts_df.columns, index=[len(televoting_final_pts_df)])

for col in votes_columns:
    new_row_sums_votes[col] = televoting_final_pts_df[col].sum()

new_row_sums_votes["FullCountryName"] = "TELEVOTING_PTS_TOTAL"
televoting_final_pts_df = pd.concat([televoting_final_pts_df, new_row_sums_votes], ignore_index=True)

televoting_final_pts_df.to_csv(os.path.join("Output", "PP-Audience-Rank.csv"))

### Stage 5h: Build the final televoting results dataframe


In [32]:
############# TELEVOTING 5 : Final DF #############

final_televoting_results_df = televoting_final_pts_df.set_index('FullCountryName')
transposed_final_televoting_results_df = final_televoting_results_df.T
transposed_final_televoting_results_df = transposed_final_televoting_results_df.reset_index()


transposed_final_televoting_results_df = transposed_final_televoting_results_df.rename(columns={'index': 'Country'})
transposed_final_televoting_results_df = transposed_final_televoting_results_df[transposed_final_televoting_results_df['Country'].isin(votes_columns)]
transposed_final_televoting_results_df = transposed_final_televoting_results_df.sort_values(by='TELEVOTING_PTS_TOTAL', ascending=False)

# Move the 'TELEVOTING_PTS_TOTAL' column to the second position
cols = transposed_final_televoting_results_df.columns.tolist()
cols.insert(1, cols.pop(cols.index('TELEVOTING_PTS_TOTAL')))
transposed_final_televoting_results_df = transposed_final_televoting_results_df[cols]

# Replace the substring 'nVotesforDDI' with 'nPointsforDDI' in the 'Country' column
transposed_final_televoting_results_df['Country'] = transposed_final_televoting_results_df['Country'].str.replace('nVotesforDDI', 'nPointsforDDI')

transposed_final_televoting_results_df

FullCountryName,Country,TELEVOTING_PTS_TOTAL,Austria,Germany,United Kingdom,Italy,France,Albania,Bulgaria,Montenegro,Croatia,Serbia,Switzerland,Estonia,Finland,Australia,Denmark,Norway,Sweden,Georgia,Azerbaijan,Armenia,Israel,Poland,Ukraine,Moldova,Romania,Luxembourg,Portugal,Czechia,Belgium,Cyprus,Greece,Lithuania,San Marino,Malta,Latvia,Rest Of World
99,nPointsforDDI19_Germany,142,12,0,12,4,12,0,0,0,0,0,7,0,0,6,10,10,6,8,0,0,12,0,10,0,6,1,0,4,8,2,2,1,1,8,0,0
94,nPointsforDDI14_Belgium,136,8,8,8,0,6,4,0,0,0,0,0,0,0,0,7,1,0,1,0,7,0,0,0,1,12,0,10,7,0,10,10,12,12,5,7,0
101,nPointsforDDI21_Poland,133,7,6,6,0,7,0,0,4,12,0,0,1,0,0,8,0,2,0,0,5,0,0,6,10,2,8,0,0,0,12,8,8,8,3,10,0
102,nPointsforDDI22_Portugal,120,5,5,5,2,5,0,0,0,0,0,12,12,0,3,3,7,10,4,12,6,0,5,1,0,0,12,0,8,0,0,0,0,0,0,0,3
86,nPointsforDDI06_Luxembourg,116,4,4,4,6,2,0,10,0,7,0,8,10,0,0,12,12,3,0,0,0,10,6,12,4,0,0,0,2,0,0,0,0,0,0,0,0
95,nPointsforDDI15_Estonia,115,6,7,7,10,4,0,2,0,2,0,10,0,0,0,0,0,4,2,0,1,6,4,0,0,8,0,8,0,0,7,7,6,6,0,8,0
82,nPointsforDDI02_Austria,100,0,12,10,0,8,6,0,0,0,8,0,0,0,0,0,0,0,10,10,3,0,0,0,0,10,0,1,0,7,3,5,3,4,0,0,0
89,nPointsforDDI09_Ukraine,99,2,2,2,7,0,0,0,8,8,6,6,2,8,4,0,8,0,0,0,0,3,0,0,5,0,0,0,0,10,0,0,0,0,10,0,8
83,nPointsforDDI03_Cyprus,98,0,0,0,0,0,3,8,0,0,0,2,0,7,7,0,0,7,0,0,0,7,0,0,2,0,0,4,0,1,0,12,10,10,6,12,0
93,nPointsforDDI13_United Kingdom,95,10,10,0,0,10,7,0,0,3,1,0,0,1,12,0,0,0,0,7,0,0,10,0,0,0,0,0,0,12,0,0,0,0,7,0,5


### Stage 5i: Attach televoting tie-breaking rank and final 1..N rank


In [33]:
############# TELEVOTING 6 : Final DF #############

transposed_final_televoting_results_w_tiebreakers_df = transposed_final_televoting_results_df
tv_tiebreaking_ranks_only_df = tv_tiebreaking_df[['Country', 'TIEBREAKING_RANK']]
transposed_final_televoting_results_w_tiebreakers_df = transposed_final_televoting_results_w_tiebreakers_df.merge(tv_tiebreaking_ranks_only_df, on='Country', how='left')

# Move the 'TELEVOTING_PTS_TOTAL' column to the second position
cols = transposed_final_televoting_results_w_tiebreakers_df.columns.tolist()
cols.insert(2, cols.pop(cols.index('TIEBREAKING_RANK')))
transposed_final_televoting_results_w_tiebreakers_df = transposed_final_televoting_results_w_tiebreakers_df[cols]

# Sort for final order
transposed_final_televoting_results_w_tiebreakers_df = transposed_final_televoting_results_w_tiebreakers_df.sort_values(by=['TELEVOTING_PTS_TOTAL', 'TIEBREAKING_RANK'], ascending=[False, True])
transposed_final_televoting_results_w_tiebreakers_df.insert(0, 'Final_Rank', range(1, len(transposed_final_televoting_results_w_tiebreakers_df) + 1))

# Save
transposed_final_televoting_results_w_tiebreakers_df.to_excel(os.path.join(SCRIPT_XLSX_DIRECTORY, '06.televoting_final_televoting_results_w_tiebreaker.xlsx'), index=False)

transposed_final_televoting_results_w_tiebreakers_df

,Final_Rank,Country,TELEVOTING_PTS_TOTAL,TIEBREAKING_RANK,Austria,Germany,United Kingdom,Italy,France,Albania,Bulgaria,Montenegro,Croatia,Serbia,Switzerland,Estonia,Finland,Australia,Denmark,Norway,Sweden,Georgia,Azerbaijan,Armenia,Israel,Poland,Ukraine,Moldova,Romania,Luxembourg,Portugal,Czechia,Belgium,Cyprus,Greece,Lithuania,San Marino,Malta,Latvia,Rest Of World
0,1,nPointsforDDI19_Germany,142,1,12,0,12,4,12,0,0,0,0,0,7,0,0,6,10,10,6,8,0,0,12,0,10,0,6,1,0,4,8,2,2,1,1,8,0,0
1,2,nPointsforDDI14_Belgium,136,5,8,8,8,0,6,4,0,0,0,0,0,0,0,0,7,1,0,1,0,7,0,0,0,1,12,0,10,7,0,10,10,12,12,5,7,0
2,3,nPointsforDDI21_Poland,133,2,7,6,6,0,7,0,0,4,12,0,0,1,0,0,8,0,2,0,0,5,0,0,6,10,2,8,0,0,0,12,8,8,8,3,10,0
3,4,nPointsforDDI22_Portugal,120,4,5,5,5,2,5,0,0,0,0,0,12,12,0,3,3,7,10,4,12,6,0,5,1,0,0,12,0,8,0,0,0,0,0,0,0,3
4,5,nPointsforDDI06_Luxembourg,116,6,4,4,4,6,2,0,10,0,7,0,8,10,0,0,12,12,3,0,0,0,10,6,12,4,0,0,0,2,0,0,0,0,0,0,0,0
5,6,nPointsforDDI15_Estonia,115,3,6,7,7,10,4,0,2,0,2,0,10,0,0,0,0,0,4,2,0,1,6,4,0,0,8,0,8,0,0,7,7,6,6,0,8,0
6,7,nPointsforDDI02_Austria,100,10,0,12,10,0,8,6,0,0,0,8,0,0,0,0,0,0,0,10,10,3,0,0,0,0,10,0,1,0,7,3,5,3,4,0,0,0
7,8,nPointsforDDI09_Ukraine,99,7,2,2,2,7,0,0,0,8,8,6,6,2,8,4,0,8,0,0,0,0,3,0,0,5,0,0,0,0,10,0,0,0,0,10,0,8
8,9,nPointsforDDI03_Cyprus,98,9,0,0,0,0,0,3,8,0,0,0,2,0,7,7,0,0,7,0,0,0,7,0,0,2,0,0,4,0,1,0,12,10,10,6,12,0
9,10,nPointsforDDI13_United Kingdom,95,15,10,10,0,0,10,7,0,0,3,1,0,0,1,12,0,0,0,0,7,0,0,10,0,0,0,0,0,0,12,0,0,0,0,7,0,5


### Quick inspection of the final televoting results


In [34]:
transposed_final_televoting_results_w_tiebreakers_df

,Final_Rank,Country,TELEVOTING_PTS_TOTAL,TIEBREAKING_RANK,Austria,Germany,United Kingdom,Italy,France,Albania,Bulgaria,Montenegro,Croatia,Serbia,Switzerland,Estonia,Finland,Australia,Denmark,Norway,Sweden,Georgia,Azerbaijan,Armenia,Israel,Poland,Ukraine,Moldova,Romania,Luxembourg,Portugal,Czechia,Belgium,Cyprus,Greece,Lithuania,San Marino,Malta,Latvia,Rest Of World
0,1,nPointsforDDI19_Germany,142,1,12,0,12,4,12,0,0,0,0,0,7,0,0,6,10,10,6,8,0,0,12,0,10,0,6,1,0,4,8,2,2,1,1,8,0,0
1,2,nPointsforDDI14_Belgium,136,5,8,8,8,0,6,4,0,0,0,0,0,0,0,0,7,1,0,1,0,7,0,0,0,1,12,0,10,7,0,10,10,12,12,5,7,0
2,3,nPointsforDDI21_Poland,133,2,7,6,6,0,7,0,0,4,12,0,0,1,0,0,8,0,2,0,0,5,0,0,6,10,2,8,0,0,0,12,8,8,8,3,10,0
3,4,nPointsforDDI22_Portugal,120,4,5,5,5,2,5,0,0,0,0,0,12,12,0,3,3,7,10,4,12,6,0,5,1,0,0,12,0,8,0,0,0,0,0,0,0,3
4,5,nPointsforDDI06_Luxembourg,116,6,4,4,4,6,2,0,10,0,7,0,8,10,0,0,12,12,3,0,0,0,10,6,12,4,0,0,0,2,0,0,0,0,0,0,0,0
5,6,nPointsforDDI15_Estonia,115,3,6,7,7,10,4,0,2,0,2,0,10,0,0,0,0,0,4,2,0,1,6,4,0,0,8,0,8,0,0,7,7,6,6,0,8,0
6,7,nPointsforDDI02_Austria,100,10,0,12,10,0,8,6,0,0,0,8,0,0,0,0,0,0,0,10,10,3,0,0,0,0,10,0,1,0,7,3,5,3,4,0,0,0
7,8,nPointsforDDI09_Ukraine,99,7,2,2,2,7,0,0,0,8,8,6,6,2,8,4,0,8,0,0,0,0,3,0,0,5,0,0,0,0,10,0,0,0,0,10,0,8
8,9,nPointsforDDI03_Cyprus,98,9,0,0,0,0,0,3,8,0,0,0,2,0,7,7,0,0,7,0,0,0,7,0,0,2,0,0,4,0,1,0,12,10,10,6,12,0
9,10,nPointsforDDI13_United Kingdom,95,15,10,10,0,0,10,7,0,0,3,1,0,0,1,12,0,0,0,0,7,0,0,10,0,0,0,0,0,0,12,0,0,0,0,7,0,5


## Final jury results (recap)


In [35]:
##########################
### FINAL JURY RESULTS ###
##########################

transposed_final_jury_results_w_tiebreakers_df

,Final_Rank,Recipient_Country,Total_without_RoW,TIEBREAKING_RANK,Total_with_RoW,Australia,Austria,Cyprus,Denmark,Latvia,Luxembourg,Norway,Romania,Ukraine,Czechia,Switzerland,France,United Kingdom,Belgium,Estonia,Finland,Greece,Lithuania,Germany,Italy,Poland,Portugal,Sweden,Serbia,Moldova,Albania,Armenia,Azerbaijan,Bulgaria,Croatia,Georgia,Israel,Malta,Montenegro,San Marino,Rest Of World,Rank_with_RoW,Rank_without_RoW
0,1,nPointsforDDI05_Latvia,154,1.0,166,3,0,0,6,0,10,12,3,1,0,3,8,0,3,3,12,5,0,12,5,0,12,12,10,0,0,0,12,5,12,0,4,1,0,0,12,NaN,NaN
1,2,nPointsforDDI01_Australia,125,2.0,131,0,4,0,0,0,0,0,12,0,3,0,1,0,0,0,4,7,0,8,6,8,8,3,2,10,3,2,8,6,0,3,0,10,10,7,6,NaN,NaN
2,3,nPointsforDDI22_Portugal,119,3.0,122,12,0,5,3,0,8,0,0,8,0,7,7,0,0,0,0,0,12,0,10,4,0,1,0,6,0,0,10,7,1,4,2,8,4,0,3,NaN,NaN
3,4,nPointsforDDI03_Cyprus,111,8.0,116,6,0,0,0,0,2,5,4,12,7,12,5,12,0,0,0,0,0,0,0,7,0,5,0,12,0,12,3,0,0,0,7,0,0,0,5,NaN,NaN
4,5,nPointsforDDI14_Belgium,101,14.0,101,0,0,0,0,8,0,7,7,4,0,0,0,5,0,0,0,8,0,0,0,0,0,10,6,0,12,0,0,10,4,12,0,0,0,8,0,NaN,NaN
5,6,nPointsforDDI20_Italy,95,4.0,95,5,5,1,12,0,0,6,0,10,12,0,0,1,0,6,0,0,6,0,0,1,6,0,3,0,5,0,0,0,0,0,1,3,12,0,0,NaN,NaN
6,7,nPointsforDDI02_Austria,93,5.0,93,0,0,0,0,0,0,0,8,7,2,1,0,3,8,7,0,12,2,0,0,0,0,2,0,0,10,0,0,3,0,8,12,0,5,3,0,NaN,NaN
7,8,nPointsforDDI16_Finland,90,12.0,92,0,8,12,0,12,0,0,1,0,0,0,6,0,12,0,0,3,0,0,0,12,2,4,8,0,0,5,1,4,0,0,0,0,0,0,2,NaN,NaN
8,9,nPointsforDDI15_Estonia,89,6.0,89,0,0,8,5,7,4,0,0,0,8,6,0,0,0,0,0,6,0,3,1,0,0,0,0,5,7,6,5,12,0,0,0,5,0,1,0,NaN,NaN
9,10,nPointsforDDI21_Poland,87,16.0,87,0,3,0,0,0,12,0,10,0,0,5,10,0,10,10,6,0,0,0,0,0,1,0,0,0,0,0,7,0,0,7,0,6,0,0,0,NaN,NaN


## Manual instruction placeholder


In [36]:
### Instruction XYZ

## Grand Final calculations

When the selected dataframe is the Grand Final, combine jury (without RoW) and televoting totals into a single sorted scoreboard, and export both the simple and enriched Excel deliverables.


In [37]:
################################
### GRAND FINAL CALCULATIONS ###
################################


# Check if we currently deal with the grand finale
if selected_df_name in ["sf1_televoting_df","sf2_televoting_df",'gf_televoting_df']:
    print(f'Final Score event detected, running analysis... (currently selected: {selected_df_name})')
    
    gf_televoting_results_w_tiebreakers_df = transposed_final_televoting_results_w_tiebreakers_df[['Country','TELEVOTING_PTS_TOTAL','TIEBREAKING_RANK']]
    gf_televoting_results_w_tiebreakers_df = gf_televoting_results_w_tiebreakers_df.rename(columns={'Country': 'Recipient_Country', 'TIEBREAKING_RANK': 'TELEVOTING_TIEBREAKING_RANK'})

    gf_jury_results_w_tiebreakers_df = transposed_final_jury_results_w_tiebreakers_df[['Recipient_Country','Total_without_RoW','TIEBREAKING_RANK']]
    gf_jury_results_w_tiebreakers_df = gf_jury_results_w_tiebreakers_df.rename(columns={'Total_without_RoW': 'JURY_PTS_TOTAL_WO_ROW','TIEBREAKING_RANK': 'JURY_TIEBREAKING_RANK'})
    gf_jury_results_w_tiebreakers_df = gf_jury_results_w_tiebreakers_df[gf_jury_results_w_tiebreakers_df['Recipient_Country'].str.startswith('nPointsforDDI')]

    merged_df = gf_televoting_results_w_tiebreakers_df.merge(gf_jury_results_w_tiebreakers_df, on='Recipient_Country')
    merged_df['ALL_PTS_TOTAL'] = merged_df['TELEVOTING_PTS_TOTAL'] + merged_df['JURY_PTS_TOTAL_WO_ROW']
    merged_df.insert(1, 'ALL_PTS_TOTAL', merged_df.pop('ALL_PTS_TOTAL'))
    merged_df = merged_df.sort_values(['ALL_PTS_TOTAL','TELEVOTING_PTS_TOTAL'], ascending=False)
    gf_final_merged_results_df = merged_df
else:
    print(f'Currently selected DF is not for Grand Finale (currently selected: {selected_df_name})')


gf_final_merged_results_df[['Recipient_Country','JURY_PTS_TOTAL_WO_ROW','TELEVOTING_PTS_TOTAL','ALL_PTS_TOTAL']].to_excel(os.path.join(
    SCRIPT_XLSX_DIRECTORY, '07a.SF1_FINAL_RESULTS_SIMPLE.xlsx'
), index=False)

gf_final_merged_results_df.to_excel(os.path.join(
    SCRIPT_XLSX_DIRECTORY, '07b.SF1_FINAL_RESULTS_ENRICHED.xlsx'
), index=False)


#gf_final_merged_results_df
gf_final_merged_results_df[['Recipient_Country','JURY_PTS_TOTAL_WO_ROW','TELEVOTING_PTS_TOTAL','ALL_PTS_TOTAL']]

Final Score event detected, running analysis... (currently selected: gf_televoting_df)


,Recipient_Country,JURY_PTS_TOTAL_WO_ROW,TELEVOTING_PTS_TOTAL,ALL_PTS_TOTAL
3,nPointsforDDI22_Portugal,119,120,239
1,nPointsforDDI14_Belgium,101,136,237
15,nPointsforDDI05_Latvia,154,74,228
2,nPointsforDDI21_Poland,87,133,220
8,nPointsforDDI03_Cyprus,111,98,209
5,nPointsforDDI15_Estonia,89,115,204
12,nPointsforDDI01_Australia,125,77,202
0,nPointsforDDI19_Germany,51,142,193
6,nPointsforDDI02_Austria,93,100,193
4,nPointsforDDI06_Luxembourg,65,116,181


## Combined final results table

Build a single combined dataframe holding, for each recipient country:

- the jury total (without RoW),
- the televoting total,
- a `FINAL_TOTAL_SCORE` summing both,
- per-donor breakdowns (`*_JURY`, `*_TV`, `*_TOTAL`).


In [38]:
###################################
### COMBINED FINAL RESULTS TABLE ##
###################################

# Build a combined dataframe from the jury and televoting tiebreaker tables,
# adding a final total score that sums points assigned by the jury and the televoting,
# and including the per-country breakdown (sum of jury + televoting points from each donor country).

# --- Identify donor-country columns in each source dataframe ---
jury_meta_cols = [
    'Final_Rank', 'Recipient_Country', 'Total_without_RoW', 'TIEBREAKING_RANK',
    'Total_with_RoW', 'Rank_with_RoW', 'Rank_without_RoW'
]
tv_meta_cols = [
    'Final_Rank', 'Country', 'TELEVOTING_PTS_TOTAL', 'TIEBREAKING_RANK'
]

jury_country_cols = [c for c in transposed_final_jury_results_w_tiebreakers_df.columns if c not in jury_meta_cols]
tv_country_cols   = [c for c in transposed_final_televoting_results_w_tiebreakers_df.columns if c not in tv_meta_cols]

# --- Prepare jury side ---
jury_side_df = transposed_final_jury_results_w_tiebreakers_df.copy()
jury_side_df = jury_side_df[jury_side_df['Recipient_Country'].str.startswith('nPointsforDDI')]
jury_side_df = jury_side_df[['Recipient_Country', 'Total_without_RoW'] + jury_country_cols].rename(
    columns={'Total_without_RoW': 'JURY_PTS_TOTAL'}
)
# RoW from the jury CSV is the Pot 0 jury substitute, NOT a real national jury.
# Per user instruction and PDF page 25, RoW must not contribute as a jury voter to
# the final per-recipient totals. Zero out its jury column here so only its 58
# audience points flow through.
if 'Rest Of World' in jury_side_df.columns:
    jury_side_df['Rest Of World'] = 0

# --- Prepare televoting side ---
tv_side_df = transposed_final_televoting_results_w_tiebreakers_df.copy()
tv_side_df = tv_side_df[['Country', 'TELEVOTING_PTS_TOTAL'] + tv_country_cols].rename(
    columns={'Country': 'Recipient_Country'}
)

# --- Merge on Recipient_Country with suffixes to keep both donor breakdowns ---
combined_final_results_df = jury_side_df.merge(
    tv_side_df, on='Recipient_Country', how='inner', suffixes=('_JURY', '_TV')
)

combined_final_results_df['FINAL_TOTAL_SCORE'] = (
    combined_final_results_df['JURY_PTS_TOTAL'] + combined_final_results_df['TELEVOTING_PTS_TOTAL']
)

# --- Build per-country combined columns: sum of jury + televoting points from each donor country ---
donor_countries = sorted(set([c.replace('Rest Of World', 'Rest Of World') for c in jury_country_cols]) | set(tv_country_cols))
for donor in donor_countries:
    j_col = donor if donor in combined_final_results_df.columns else f'{donor}_JURY'
    t_col = donor if donor in combined_final_results_df.columns else f'{donor}_TV'
    j_vals = pd.to_numeric(combined_final_results_df[j_col], errors='coerce').fillna(0) if j_col in combined_final_results_df.columns else 0
    t_vals = pd.to_numeric(combined_final_results_df[t_col], errors='coerce').fillna(0) if t_col in combined_final_results_df.columns else 0
    combined_final_results_df[f'{donor}_TOTAL'] = j_vals + t_vals

# --- Sort and rank ---
combined_final_results_df = combined_final_results_df.sort_values(
    by=['FINAL_TOTAL_SCORE', 'TELEVOTING_PTS_TOTAL'], ascending=[False, False]
).reset_index(drop=True)
combined_final_results_df.insert(0, 'Final_Rank', range(1, len(combined_final_results_df) + 1))

# --- Reorder: meta columns first, then per-country totals, then jury/televoting breakdowns ---
total_country_cols = [f'{c}_TOTAL' for c in donor_countries]
jury_only_cols = [c for c in combined_final_results_df.columns if c.endswith('_JURY')]
tv_only_cols   = [c for c in combined_final_results_df.columns if c.endswith('_TV')]
shared_cols    = [c for c in donor_countries if c in combined_final_results_df.columns]

ordered_cols = (
    ['Final_Rank', 'Recipient_Country', 'FINAL_TOTAL_SCORE', 'JURY_PTS_TOTAL', 'TELEVOTING_PTS_TOTAL']
    + total_country_cols
    + jury_only_cols + tv_only_cols + shared_cols
)
ordered_cols = [c for c in ordered_cols if c in combined_final_results_df.columns]
combined_final_results_df = combined_final_results_df[ordered_cols]

combined_final_results_df

,Final_Rank,Recipient_Country,FINAL_TOTAL_SCORE,JURY_PTS_TOTAL,TELEVOTING_PTS_TOTAL,Albania_TOTAL,Armenia_TOTAL,Australia_TOTAL,Austria_TOTAL,Azerbaijan_TOTAL,Belgium_TOTAL,Bulgaria_TOTAL,Croatia_TOTAL,Cyprus_TOTAL,Czechia_TOTAL,Denmark_TOTAL,Estonia_TOTAL,Finland_TOTAL,France_TOTAL,Georgia_TOTAL,Germany_TOTAL,Greece_TOTAL,Israel_TOTAL,Italy_TOTAL,Latvia_TOTAL,Lithuania_TOTAL,Luxembourg_TOTAL,Malta_TOTAL,Moldova_TOTAL,Montenegro_TOTAL,Norway_TOTAL,Poland_TOTAL,Portugal_TOTAL,Rest Of World_TOTAL,Romania_TOTAL,San Marino_TOTAL,Serbia_TOTAL,Sweden_TOTAL,Switzerland_TOTAL,Ukraine_TOTAL,United Kingdom_TOTAL,Australia_JURY,Austria_JURY,Cyprus_JURY,Denmark_JURY,Latvia_JURY,Luxembourg_JURY,Norway_JURY,Romania_JURY,Ukraine_JURY,Czechia_JURY,Switzerland_JURY,France_JURY,United Kingdom_JURY,Belgium_JURY,Estonia_JURY,Finland_JURY,Greece_JURY,Lithuania_JURY,Germany_JURY,Italy_JURY,Poland_JURY,Portugal_JURY,Sweden_JURY,Serbia_JURY,Moldova_JURY,Albania_JURY,Armenia_JURY,Azerbaijan_JURY,Bulgaria_JURY,Croatia_JURY,Georgia_JURY,Israel_JURY,Malta_JURY,Montenegro_JURY,San Marino_JURY,Rest Of World_JURY,Austria_TV,Germany_TV,United Kingdom_TV,Italy_TV,France_TV,Albania_TV,Bulgaria_TV,Montenegro_TV,Croatia_TV,Serbia_TV,Switzerland_TV,Estonia_TV,Finland_TV,Australia_TV,Denmark_TV,Norway_TV,Sweden_TV,Georgia_TV,Azerbaijan_TV,Armenia_TV,Israel_TV,Poland_TV,Ukraine_TV,Moldova_TV,Romania_TV,Luxembourg_TV,Portugal_TV,Czechia_TV,Belgium_TV,Cyprus_TV,Greece_TV,Lithuania_TV,San Marino_TV,Malta_TV,Latvia_TV,Rest Of World_TV
0,1,nPointsforDDI22_Portugal,239,119,120,0,6,15,5,22,0,7,1,5,8,6,12,0,12,8,5,0,2,12,0,12,20,8,6,4,7,9,0,3,0,0,0,11,19,9,5,12,0,5,3,0,8,0,0,8,0,7,7,0,0,0,0,0,12,0,10,4,0,1,0,6,0,0,10,7,1,4,2,8,4,0,0,5,5,5,2,5,0,0,0,0,0,12,12,0,3,3,7,10,4,12,6,0,5,1,0,0,12,0,8,0,0,0,0,0,0,0,3
1,2,nPointsforDDI14_Belgium,237,101,136,16,7,0,8,0,0,10,4,10,7,7,0,0,6,13,8,18,0,0,15,12,0,5,1,0,8,0,10,0,19,20,6,10,0,4,13,0,0,0,0,8,0,7,7,4,0,0,0,5,0,0,0,8,0,0,0,0,0,10,6,0,12,0,0,10,4,12,0,0,0,8,0,8,8,8,0,6,4,0,0,0,0,0,0,0,0,7,1,0,1,0,7,0,0,0,1,12,0,10,7,0,10,10,12,12,5,7,0
2,3,nPointsforDDI05_Latvia,228,154,74,5,0,13,0,12,6,8,17,0,12,6,3,12,9,0,12,5,4,5,0,0,10,1,12,0,16,3,12,0,3,0,14,24,3,1,0,3,0,0,6,0,10,12,3,1,0,3,8,0,3,3,12,5,0,12,5,0,12,12,10,0,0,0,12,5,12,0,4,1,0,0,0,0,0,0,0,1,5,3,0,5,4,0,0,0,10,0,4,12,0,0,0,0,3,0,12,0,0,0,12,3,0,0,0,0,0,0,0
3,4,nPointsforDDI21_Poland,220,87,133,0,5,0,10,7,10,0,12,12,0,8,11,6,17,7,6,8,0,0,10,8,20,9,10,4,0,0,1,0,12,8,0,2,5,6,6,0,3,0,0,0,12,0,10,0,0,5,10,0,10,10,6,0,0,0,0,0,1,0,0,0,0,0,7,0,0,7,0,6,0,0,0,7,6,6,0,7,0,0,4,12,0,0,1,0,0,8,0,2,0,0,5,0,0,6,10,2,8,0,0,0,12,8,8,8,3,10,0
4,5,nPointsforDDI03_Cyprus,209,111,98,3,12,13,0,3,1,8,0,0,7,0,0,7,5,0,0,12,14,0,12,10,2,6,14,0,5,7,4,0,4,10,0,12,14,12,12,6,0,0,0,0,2,5,4,12,7,12,5,12,0,0,0,0,0,0,0,7,0,5,0,12,0,12,3,0,0,0,7,0,0,0,0,0,0,0,0,0,3,8,0,0,0,2,0,7,7,0,0,7,0,0,0,7,0,0,2,0,0,4,0,1,0,12,10,10,6,12,0
5,6,nPointsforDDI15_Estonia,204,89,115,7,7,0,6,5,0,14,2,15,8,5,0,0,4,2,10,13,6,11,15,6,4,5,5,0,0,4,8,0,8,7,0,4,16,0,7,0,0,8,5,7,4,0,0,0,8,6,0,0,0,0,0,6,0,3,1,0,0,0,0,5,7,6,5,12,0,0,0,5,0,1,0,6,7,7,10,4,0,2,0,2,0,10,0,0,0,0,0,4,2,0,1,6,4,0,0,8,0,8,0,0,7,7,6,6,0,8,0
6,7,nPointsforDDI01_Australia,202,125,77,5,2,0,4,11,0,6,10,0,8,0,4,9,1,15,8,8,0,11,0,0,3,22,10,10,0,8,14,6,15,7,2,3,0,0,0,0,4,0,0,0,0,0,12,0,3,0,1,0,0,0,4,7,0,8,6,8,8,3,2,10,3,2,8,6,0,3,0,10,10,7,0,0,0,0,5,0,2,0,0,10,0,0,4,5,0,0,0,0,12,3,0,0,0,0,0,3,3,6,5,0,0,1,0,0,12,0,6
7,8,nPointsforDDI19_Germany,193,51,142,0,0,6,13,0,12,0,0,2,4,11,0,5,12,13,0,2,12,4,0,6,8,8,0,1,10,0,0,0,6,1,5,12,11,10,19,0,1,0,1,0,7,0,0,0,0,4,0,7,4,0,5,0,5,0,0,0,0,6,5,0,0,0,0,0,0,5,0,0,1,0,0,12,0,12,4,12,0,0,0,0,0,7,0,0,6,10,10,6,8,0,0,12,0,10,0,6,1,0,4,8,2,2,1,1,8,0,0
8,9,nPointsforDDI02_Austria,193,93,100,16,3,0,0,10,15,3,0,3,2,0,7,0,8,18,12,17,12,0,0,5,0,0,0,5,0,0,1,0,18,7,8,2,1,7,13,0,0,0,0,0,0,0,8,7,2,1,0,3,8,7,0,12,2,0,0,0,0,2,0,0,10,0,0,3,0,8,12,0,5,3,0,0,12,10,0,8,6,0,0,0,8,0,0,0,0,0,0,0,10,10,3,0,0,0,0,10,0,1,0,7,3,5,3,4,0,0,0
9,10,nPointsforDDI0

### Export the combined final results to Excel


In [39]:
combined_final_results_df.to_excel(os.path.join(
    SCRIPT_XLSX_DIRECTORY, '07b.SF1_FINAL_RESULTS_ENRICHED-all.xlsx'
), index=False)